# 19 — Spatial CountRegression preparation (V3)

Preparation only, under `workflow/spatial-model-protocol-v3-draft.md`: geometry-only DOH 2000 m universe, sparse SELECTED+LINKED recorded-count targets (daily and 6 h, Asia/Bangkok), prospective geometry / calendar / coarse context features, and leakage evidence. **No model is trained, no negative sampling, no score, threshold or colour.**

- Label = number of retained 2021–25 records with `status_50m == matched AND segment_status == assigned` on a universe segment. A zero is exact only inside this conditional population; it is not zero real accidents. Unknown records stay separate.
- Features for notebook 20 (training/evaluation) and 22 (serving) come from **one** builder: `load_prepared_context(data_dir, expected_hashes)` + `build_feature_rows(ctx, requests, unit, variant)`.
- Run with `.venv/bin/python backend/scripts/run_spatial_preparation.py --output-dir backend/rerun-spatial-prep-<id>` (new root only). Row-level outputs go to the gitignored data dir; the report dir holds aggregates only.


In [1]:
# PARAMETERS (explicit; the runner sets them through environment variables, VS Code users may edit them here)
import os
from pathlib import Path


def _default_backend(c):
    for cand in (c, c.parent, c / 'backend'):
        if (cand / 'config' / 'spatial_preparation_v3.json').is_file():
            return cand
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_NAME = os.environ.get('ROADAWARE_STAGE_CONFIG') or 'spatial_preparation_v3.json'  # *_v4.json = national
if Path(CONFIG_NAME).name != CONFIG_NAME or not CONFIG_NAME.startswith('spatial_preparation_v') or not CONFIG_NAME.endswith('.json'):
    raise ValueError('ROADAWARE_STAGE_CONFIG must be a spatial_preparation_v*.json file name in backend/config')
CONFIG_PATH = BACKEND / 'config' / CONFIG_NAME
print('backend:', BACKEND, '| output root:', OUTPUT_ROOT)


backend: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend | output root: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/rerun-spatial-prep-015-002


In [2]:
# FUNCTIONS (1/6): errors, hashing, atomic writes, output guards, config validation, fail-closed input pins and the verified notebook-10 helper loader. Pure logic; the tests exec these cells.
import datetime as dt
import gzip
import hashlib
import importlib
import importlib.util
import io
import json
import math
import os
import platform
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

NL = chr(10)
UNITS = ('daily', '6h')


class PreparationError(RuntimeError):
    pass


class RequestError(ValueError):
    """A feature request is invalid or outside the supported universe: nothing is returned (never a healthy fake zero)."""

    def __init__(self, code, message, rows=None):
        super().__init__(code + ': ' + message)
        self.code, self.rows = code, list(rows or [])


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def atomic_write_bytes(path, data):
    # sibling temp file + rename: a reader never sees a half-written file
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'wb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None (allow_nan=False keeps the JSON strict)
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in (sorted(o) if isinstance(o, set) else o)]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o


def atomic_write_json(path, obj):
    atomic_write_bytes(path, (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8'))


def csv_bytes(df, gz=False):
    # floats are written with Python repr (shortest round-trip); read back with read_numeric_csv (float_precision='round_trip')
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise PreparationError('refusing to write infinite values')
    b = df.to_csv(index=False, lineterminator=NL).encode('utf-8')
    return gzip.compress(b, mtime=0) if gz else b


def read_numeric_csv(path, str_cols=()):
    """THE central numeric loader of every notebook-19 table: listed columns stay text (IDs, route codes, dates, blocks), everything else is
    parsed with float_precision='round_trip' so a float written by csv_bytes reloads bit-identically. No rounding hacks."""
    return pd.read_csv(path, dtype={c: str for c in str_cols}, keep_default_na=False, na_values=[''], float_precision='round_trip')


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(names, pins):
    missing = [n for n in names if importlib.util.find_spec(n) is None]
    if missing:
        raise PreparationError('missing Python package(s) ' + ', '.join(missing) + '; ask Codex to install ' +
                               ' '.join(pins.get(n, n) for n in missing) + '. Nothing was written.')
    return True


def as_flag(values):
    # producers write booleans as text; only an explicit true counts (missing is not true)
    return pd.Series(list(values), dtype=object).astype(str).str.strip().str.lower().isin(['true', '1', 'yes']).to_numpy()


def integral_ids(values, label):
    """Finite integral identity (e.g. '8', '8.0', 8.0) -> int64; anything else raises. Never joins formatted strings."""
    v = pd.to_numeric(pd.Series(list(values), dtype=object), errors='coerce').to_numpy(dtype=float)
    bad = ~np.isfinite(v) | (v != np.round(v))
    if bad.any():
        raise PreparationError(label + ': ' + str(int(bad.sum())) + ' non-finite or non-integral identity values')
    return v.astype(np.int64)


def norm_numeric_code(raw, regex, pad):
    """Strict DOH numeric route / control-section code: NFKC + strip, ASCII digits only (1..pad), zero-padded ('1' -> '0001', '102' -> '0102'). Else None."""
    if raw is None or (isinstance(raw, float) and not math.isfinite(raw)):
        return None
    s = unicodedata.normalize('NFKC', str(raw)).strip()
    return s.zfill(int(pad)) if re.fullmatch(regex, s) and s.isascii() else None


# ---------- configuration ----------
def validate_config(cfg):
    c = cfg['crs']
    if c['geographic_epsg'] != 4326 or c['metric_epsg'] != 32647 or c['always_xy'] is not True:
        raise PreparationError('CRS must be EPSG:4326 -> EPSG:32647 with always_xy=True')
    u = cfg['universe']
    if u['source_id'] != 'doh_highway' or int(u['resolution_m']) != 2000 or float(u['min_length_m']) != 500.0 or u['route_status'] != 'standard':
        raise PreparationError('universe predicate differs from the preregistered protocol (DOH, standard route, 2000 m, >= 500 m)')
    lb = cfg['labels']
    if (lb['primary_status_column'], lb['primary_status_value'], lb['segment_status_column'], lb['segment_status_value']) != \
            ('status_50m', 'matched', 'segment_status', 'assigned'):
        raise PreparationError('label filter must be status_50m == matched AND segment_status == assigned (never 100 m only)')
    t = cfg['time']
    if t['time_blocks'] != ['00-06', '06-12', '12-18', '18-24'] or t['timezone'] != 'Asia/Bangkok' or int(t['utc_offset_seconds']) != 25200:
        raise PreparationError('time blocks / timezone must be the protocol 6 h Asia/Bangkok blocks')
    if not (t['research_start'] <= t['population_start'] <= t['population_end'] <= t['evaluated_end'] <= t['research_end']):
        raise PreparationError('inconsistent date ranges')
    years = list(range(int(t['population_start'][:4]), int(t['population_end'][:4]) + 1))
    if lb['years'] != years:
        raise PreparationError('label years must equal the population date axis years')
    if int(cfg['batch']['max_rows']) > 100000 or int(cfg['batch']['population_batch_rows']) > 100000:
        raise PreparationError('batches must be <= 100000 rows')
    if set(cfg['features']['variants']) != {'calendar_geometry', 'calendar_geometry_context'}:
        raise PreparationError('feature variants must be calendar_geometry and calendar_geometry_context')
    for k, v in cfg['inputs'].items():
        if isinstance(v, dict) and 'path' in v and not (v.get('sha256') or v.get('functions_sha256')):
            raise PreparationError('input ' + k + ' has no sha256 pin (null pins fail closed)')
    return True


# ---------- output guards ----------
def output_paths(out_root, cfg):
    out_root = Path(out_root)
    rep, data = out_root / cfg['outputs']['report_dir'], out_root / cfg['outputs']['data_dir']
    return {'out_root': out_root, 'report_dir': rep, 'figure_dir': rep / 'figures', 'data_dir': data}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    # child is parent or beneath it: by resolved path (symlinks, ..) or by file identity of an existing ancestor (aliases)
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def guarded_paths(backend, cfg):
    backend = Path(backend)
    out = [backend / rel for rel in cfg['protected_paths']]
    out += [backend / v['path'] for v in cfg['inputs'].values() if isinstance(v, dict) and 'path' in v]
    out += [backend / cfg['protocol']['path'], backend / cfg['notebook'], backend / 'config' / CONFIG_NAME]
    return out


def check_outputs(paths, backend, cfg):
    # call before any mkdir/write: no overlap (either direction, symlinks resolved) with protected or read inputs; both dirs must be NEW
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps protected ' + str(p))
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise PreparationError('refusing output root (nothing written): ' + '; '.join(hits))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if os.path.lexists(paths[k])]
    if used:
        raise PreparationError('output already exists in ' + ', '.join(used) + '; canonical outputs are new-only: use a NEW root, e.g. backend/rerun-spatial-prep-<id>')
    return True


def check_report_file(rel, cfg):
    # the tracked report dir receives aggregated evidence only (no GeoJSON / gz / row-level files)
    if Path(rel).suffix.lower() not in cfg['report_allowed_suffixes'] or str(rel).endswith('.gz'):
        raise PreparationError('refusing to write ' + str(rel) + ' into the report dir (aggregated csv/png/md/json only)')
    return True


def hash_paths(root, rels):
    # relative path -> sha256 for every file under each protected rel; a missing rel is recorded as None
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


def protected_rels(cfg):
    return sorted(set(cfg['protected_paths']) | {v['path'] for v in cfg['inputs'].values() if isinstance(v, dict) and 'path' in v})


# ---------- input pins ----------
def verify_pinned(path, expected, label):
    path = Path(path)
    if not expected:
        raise PreparationError(label + ': no sha256 pin (fail closed)')
    if not path.is_file():
        raise PreparationError(label + ': missing input ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise PreparationError(label + ': sha256 ' + got[:12] + ' != pinned ' + str(expected)[:12] + ' (' + str(path) + '); input changed or tampered')
    return got


def producer_hashes(manifest, name):
    """Every sha256 the producer manifest records for a file called `name` (dict values keyed by name, or entries whose path/preflight_path ends in it)."""
    hits = []

    def walk(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k == name and isinstance(v, str):
                    hits.append(v)
            pth = o.get('path') or o.get('preflight_path')
            if isinstance(pth, str) and Path(pth).name == name and isinstance(o.get('sha256'), str):
                hits.append(o['sha256'])
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(manifest)
    return hits


def verify_inputs(backend, cfg):
    """Hash-verify every pinned input, then require the producer manifest to record exactly the same hash. Returns (rows, docs)."""
    backend, I, rows, docs = Path(backend), cfg['inputs'], [], {}
    proto = cfg['protocol']
    rows.append({'input': 'protocol_v3', 'path': proto['path'], 'sha256': verify_pinned(backend / proto['path'], proto['sha256'], 'protocol_v3'),
                 'rule': 'manager pin', 'status': 'pass'})
    for key in ('linkage_manifest', 'linkage_review', 'road_manifest', 'sources_manifest', 'preparation_v2_manifest', 'feature_config_v2', 'sources_config_v2'):
        v = I[key]
        h = verify_pinned(backend / v['path'], v['sha256'], key)
        docs[key] = json.loads((backend / v['path']).read_text('utf-8'))
        rows.append({'input': key, 'path': v['path'], 'sha256': h, 'rule': 'manager pin', 'status': 'pass'})
    lm, rv = docs['linkage_manifest'], docs['linkage_review']
    if lm.get('status') != I['linkage_manifest']['expect_status'] or lm.get('version') != I['linkage_manifest']['expect_version']:
        raise PreparationError('linkage manifest is not the accepted complete spatial_linkage_v3.0 run')
    if not all(c.get('status') == 'pass' for c in lm.get('checks', [])) or lm.get('protected_unchanged') is not True:
        raise PreparationError('linkage manifest records a failed check or changed protected files')
    if rv.get('status') != I['linkage_review']['expect_status'] or rv.get('all_checks_pass') is not True:
        raise PreparationError('linkage independent review is not accepted')
    prod_map = {'linkage_manifest': lm.get('data_files_gitignored', {}), 'road_manifest': docs['road_manifest'].get('derived_files', {}),
                'sources_manifest': docs['sources_manifest']}
    for key, v in I.items():
        if not isinstance(v, dict) or 'producer' not in v:
            continue
        h = verify_pinned(backend / v['path'], v['sha256'], key)
        if 'bytes' in v and (backend / v['path']).stat().st_size != int(v['bytes']):
            raise PreparationError(key + ': byte size differs from the pin')
        name = v.get('manifest_key') or Path(v['path']).name
        rec = producer_hashes(prod_map[v['producer']], name)
        if not rec or any(r != h for r in rec):
            raise PreparationError(key + ': producer ' + v['producer'] + ' records ' + str(rec) + ' for ' + name + ', not the pinned ' + h[:12])
        rows.append({'input': key, 'path': v['path'], 'sha256': h, 'rule': 'manager pin == ' + v['producer'], 'status': 'pass'})
    return pd.DataFrame(rows), docs


# ---------- verified notebook-10 helpers (FUNCTIONS cells only) ----------
def nb10_functions_source(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    return [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]


def load_nb10_helpers(nb_path, expected_sha, expected_cells=2, prep_manifest=None):
    """exec ONLY the accepted notebook-10 '# FUNCTIONS' cells after their concatenated sha256 equals the manager pin (and the V2 preparation
    manifest's functions_cells_sha256 when given). No pipeline cell, no model/registry is touched. Returns the helper namespace."""
    hits = nb10_functions_source(nb_path)
    got = sha256_bytes('\n\n'.join(hits).encode('utf-8'))
    if len(hits) != int(expected_cells) or got != expected_sha:
        raise PreparationError('notebook 10 FUNCTIONS sha256 ' + got[:12] + ' (' + str(len(hits)) + ' cells) != pinned ' + str(expected_sha)[:12])
    if prep_manifest is not None and prep_manifest.get('functions_cells_sha256') != got:
        raise PreparationError('notebook 10 FUNCTIONS sha256 differs from the accepted V2 preparation manifest')
    ns = {'__name__': 'nb10_functions'}
    for i, s in enumerate(hits):
        exec(compile(s, 'notebook10:FUNCTIONS[' + str(i) + ']', 'exec'), ns)
    for name in ('core_date_features', 'festival_features', 'season_features', 'bank_calendar_features', 'load_calendar_records',
                 'weather_features', 'validate_feature_columns', 'DependencyError'):
        if name not in ns:
            raise PreparationError('notebook 10 helper ' + name + ' missing')
    return ns, got


In [3]:
# FUNCTIONS (2/6): geometry-only universe from the accepted 18 inventory + pinned 2000 m GeoJSON, static geometry features, geoBoundaries ADM1 registry and midpoint province join.
INV_STR_COLS = ['segment_id', 'snapshot_tag', 'source_id', 'route_raw', 'route_key', 'route_status']
STATIC_FEATURES = ['route_code', 'geom_mid_x_m', 'geom_mid_y_m', 'geom_mid_lon', 'geom_mid_lat', 'geom_length_m', 'geom_log_length_m',
                   'geom_bearing_sin', 'geom_bearing_cos', 'geom_straightness', 'geom_turning_rad_per_km']


def make_transformers(cfg):
    from pyproj import Transformer
    c = cfg['crs']
    fwd = Transformer.from_crs(c['geographic_epsg'], c['metric_epsg'], always_xy=True)
    inv = Transformer.from_crs(c['metric_epsg'], c['geographic_epsg'], always_xy=True)
    return fwd, inv


def load_inventory(path):
    inv = pd.read_csv(path, dtype={c: str for c in INV_STR_COLS}, keep_default_na=False, compression='gzip')
    miss = [c for c in ['segment_id', 'resolution_m', 'snapshot_tag', 'source_id', 'oid', 'part', 'piece', 'seg_index', 'length_m',
                        'length_geod_m', 'short_final', 'route_key', 'route_status'] if c not in inv.columns]
    if miss:
        raise PreparationError('segment inventory missing columns ' + ', '.join(miss))
    if (inv['segment_id'].str.strip() == '').any() or inv['segment_id'].duplicated().any():
        raise PreparationError('segment inventory has empty or duplicate segment_id')
    for c in ('oid', 'part', 'piece', 'seg_index'):
        inv[c] = integral_ids(inv[c], 'inventory ' + c)
    for c in ('resolution_m', 'length_m', 'length_geod_m'):
        inv[c] = pd.to_numeric(inv[c], errors='raise').astype(float)
    return inv


def load_segment_geojson(path):
    fc = json.loads(Path(path).read_text('utf-8'))
    ids = [f.get('id') for f in fc.get('features', [])]
    if any(not isinstance(i, str) or not i for i in ids) or len(set(ids)) != len(ids):
        raise PreparationError('segment GeoJSON has missing or duplicate feature ids')
    out = {}
    for f in fc['features']:
        g = f.get('geometry') or {}
        c = np.asarray(g.get('coordinates', []), float)
        if g.get('type') != 'LineString' or c.ndim != 2 or c.shape[0] < 2 or c.shape[1] != 2 or not np.isfinite(c).all():
            raise PreparationError('segment ' + f['id'] + ' geometry is not a finite 2-D LineString')
        out[f['id']] = (c, f.get('properties') or {})
    return out


def classify_segments(inv, cfg):
    """Geometry-only predicate (no outcome used). Returns (universe rows, unsupported rows with the first failing reason)."""
    u = cfg['universe']
    R = inv[inv['resolution_m'] == float(u['resolution_m'])].copy()
    reason = np.full(len(R), '', dtype=object)
    rules = [(R['snapshot_tag'] != cfg['snapshot_tag'], 'snapshot_mismatch'), (R['source_id'] != u['source_id'], 'unsupported_source_' + R['source_id']),
             (R['route_status'] != u['route_status'], 'route_' + R['route_status']), (R['length_m'] < float(u['min_length_m']), 'short_lt_min_length')]
    for m, lab in rules:
        m = m.to_numpy()
        lab = lab.to_numpy() if hasattr(lab, 'to_numpy') else np.full(len(R), lab, dtype=object)
        reason = np.where((reason == '') & m, lab, reason)
    R['unsupported_reason'] = reason
    uni = R[reason == ''].drop(columns=['unsupported_reason']).sort_values('segment_id').reset_index(drop=True)
    uns = R[reason != ''][['segment_id', 'source_id', 'route_key', 'route_status', 'length_m', 'unsupported_reason']].sort_values('segment_id').reset_index(drop=True)
    if (uni['snapshot_tag'] != cfg['snapshot_tag']).any():
        raise PreparationError('universe snapshot tag differs from the pinned snapshot')
    if uni.duplicated(['source_id', 'oid', 'part', 'piece', 'seg_index']).any():
        raise PreparationError('duplicate geometry identity (source/OID/part/piece/index) in the universe')
    return uni, uns


def turning_per_km(xy, length_m, eps):
    # sum of absolute turning angles between consecutive non-degenerate edges (rad) per km; zero-length edges are skipped (finite, documented)
    d = np.diff(xy, axis=0)
    d = d[np.hypot(d[:, 0], d[:, 1]) > eps]
    if len(d) < 2:
        return 0.0
    a, b = d[:-1], d[1:]
    ang = np.abs(np.arctan2(a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0], (a * b).sum(axis=1)))
    return float(ang.sum() / (length_m / 1000.0))


def static_geometry(uni, geo, cfg, tr):
    """Static features from the pinned 2000 m GeoJSON only (never event coordinates). Projected lengths must reproduce the inventory."""
    from pyproj import Geod
    from shapely.geometry import LineString
    fwd, inv = tr
    geod, eps, tol = Geod(ellps='WGS84'), float(cfg['features']['chord_zero_eps_m']), float(cfg['universe']['length_tol_m'])
    rows, bad = [], []
    for r in uni.itertuples(index=False):
        if r.segment_id not in geo:
            raise PreparationError('universe segment ' + r.segment_id + ' missing from the pinned GeoJSON')
        ll, props = geo[r.segment_id]
        x, y = fwd.transform(ll[:, 0], ll[:, 1])
        xy = np.column_stack([x, y])
        L = float(np.hypot(np.diff(x), np.diff(y)).sum())
        G = float(geod.line_length(ll[:, 0], ll[:, 1]))
        if abs(L - r.length_m) > tol or abs(G - r.length_geod_m) > tol or str(props.get('segment_id', r.segment_id)) != r.segment_id:
            bad.append(r.segment_id)
        if not L > 0:
            raise PreparationError(r.segment_id + ': non-positive projected length')
        line = LineString(xy)
        mid = line.interpolate(0.5, normalized=True)
        mlon, mlat = inv.transform(mid.x, mid.y)
        dx, dy = xy[-1, 0] - xy[0, 0], xy[-1, 1] - xy[0, 1]
        chord = float(math.hypot(dx, dy))
        zero = chord <= eps
        b = 0.0 if zero else math.atan2(dx, dy)  # grid azimuth clockwise from UTM north, start -> end in digitised direction
        rows.append({'segment_id': r.segment_id, 'route_code': r.route_key, 'geom_mid_x_m': float(mid.x), 'geom_mid_y_m': float(mid.y),
                     'geom_mid_lon': float(mlon), 'geom_mid_lat': float(mlat), 'geom_length_m': L, 'geom_log_length_m': float(math.log(L)),
                     'geom_bearing_sin': 0.0 if zero else float(math.sin(b)), 'geom_bearing_cos': 0.0 if zero else float(math.cos(b)),
                     'geom_straightness': min(1.0, chord / L), 'geom_turning_rad_per_km': turning_per_km(xy, L, eps),
                     'chord_zero_guard': bool(zero), 'length_geod_m': G, 'utm_over_geodesic': L / G if G > 0 else np.nan,
                     'oid': int(r.oid), 'part': int(r.part), 'piece': int(r.piece), 'seg_index': int(r.seg_index), 'short_final': r.short_final,
                     'piece_overlap': getattr(r, 'piece_overlap', ''), 'piece_duplicate': getattr(r, 'piece_duplicate', ''), 'coords_ll': ll})
    if bad:
        raise PreparationError(str(len(bad)) + ' segments do not reproduce inventory lengths/ids from the pinned GeoJSON (e.g. ' + ', '.join(bad[:3]) + ')')
    S = pd.DataFrame(rows)
    num = S[[c for c in STATIC_FEATURES if c != 'route_code']].to_numpy(float)
    if not np.isfinite(num).all():
        raise PreparationError('non-finite static geometry feature')
    return S


def doh_identity(doh_full_path, oids, route_keys, cfg):
    """sourceOID -> DOH road_code / section_co (accepted 17 full GeoJSON). OIDs are validated finite integers; codes strictly normalised."""
    fc = json.loads(Path(doh_full_path).read_text('utf-8'))
    t = cfg['traffic']
    ids = integral_ids([f.get('id') for f in fc['features']], 'DOH feature id')
    if len(set(ids.tolist())) != len(ids):
        raise PreparationError('duplicate DOH feature ids')
    idx = {}
    for i, f in zip(ids.tolist(), fc['features']):
        p = f.get('properties') or {}
        if p.get('objectid') is not None and int(integral_ids([p['objectid']], 'DOH objectid')[0]) != i:
            raise PreparationError('DOH feature id ' + str(i) + ' differs from its objectid property')
        idx[i] = (norm_numeric_code(p.get('road_code'), t['route_regex'], t['pad']), norm_numeric_code(p.get('section_co'), t['route_regex'], t['pad']))
    rows = []
    for o, rk in zip(oids, route_keys):
        if int(o) not in idx:
            raise PreparationError('universe OID ' + str(o) + ' absent from the DOH full GeoJSON')
        rc, sc = idx[int(o)]
        if rc != rk:
            raise PreparationError('OID ' + str(o) + ': DOH road_code ' + str(rc) + ' differs from the inventory route_key ' + str(rk))
        rows.append({'aadt_route': rc, 'aadt_section': sc, 'aadt_identity_status': 'ok' if sc else 'section_code_nonstandard_or_missing'})
    return pd.DataFrame(rows)


# ---------- geoBoundaries ADM1 registry (registered BEFORE any midpoint join) ----------
def load_admin(geometry_path, metadata_path, points_path, cfg, geometry_sha):
    from shapely.geometry import shape
    a = cfg['admin']
    meta = json.loads(Path(metadata_path).read_text('utf-8'))
    meta = meta[0] if isinstance(meta, list) else meta
    diff = {k: meta.get(k) for k, v in a['expect_metadata'].items() if str(meta.get(k)) != v}
    if diff:
        raise PreparationError('geoBoundaries metadata differs from the registered expectation: ' + str(diff))
    fc = json.loads(Path(geometry_path).read_text('utf-8'))
    rows, polys = [], []
    for f in fc.get('features', []):
        p = f.get('properties') or {}
        g = shape(f['geometry'])
        rows.append({'shape_iso': str(p.get(a['iso_property'], '')), 'shape_id': str(p.get(a['id_property'], '')), 'shape_name': str(p.get(a['name_property'], '')),
                     'geom_type': g.geom_type, 'is_valid': bool(g.is_valid), 'is_empty': bool(g.is_empty)})
        polys.append(g)
    P = pd.DataFrame(rows)
    n = int(a['expected_polygons'])
    problems = []
    if len(P) != n:
        problems.append(str(len(P)) + ' polygons, expected ' + str(n))
    if P['shape_iso'].duplicated().any() or P['shape_id'].duplicated().any() or (P['shape_iso'] == '').any():
        problems.append('duplicate or empty ISO/ID')
    if (~P['is_valid']).any() or P['is_empty'].any() or (~P['geom_type'].isin(['Polygon', 'MultiPolygon'])).any():
        problems.append('invalid/empty/non-polygon geometry (never repaired): ' + ', '.join(P.loc[~P['is_valid'] | P['is_empty'], 'shape_iso'].tolist()[:5]))
    pts = pd.read_csv(points_path, dtype=str, keep_default_na=False)
    if len(pts) != n or pts['province_code'].duplicated().any() or pts['shape_iso'].duplicated().any() or pts['province_key'].duplicated().any() \
            or (pts['province_key'].str.strip() == '').any():
        problems.append('province_points must hold ' + str(n) + ' unique province_code/shape_iso/province_key rows')
    code = integral_ids(pts['province_code'], 'province_code')
    if not (pts['shape_iso'] == ['TH-' + str(c) for c in code]).all():
        problems.append('ISO 3166-2 code does not equal TH-<province_code>')
    if set(pts['shape_iso']) != set(P['shape_iso']):
        problems.append('province_points ISO set differs from the polygons')
    j = pts.merge(P, on='shape_iso', how='left', suffixes=('', '_poly'))
    if (j['shape_id'] != j['shape_id_poly']).any():
        problems.append('province_points shape_id differs from the polygon shapeID')
    if (pts['geometry_sha256'] != geometry_sha).any():
        problems.append('province_points were not derived from the pinned polygon file')
    if problems:
        raise PreparationError('ADM1 registry failed: ' + '; '.join(problems))
    P['province_code'] = P['shape_iso'].map(dict(zip(pts['shape_iso'], code.tolist())))
    P['province_key'] = P['shape_iso'].map(dict(zip(pts['shape_iso'], pts['province_key'])))
    return {'table': P, 'polys': polys, 'metadata': {k: meta.get(k) for k in ('boundaryID', 'boundaryYearRepresented', 'boundarySource', 'boundaryLicense',
                                                                             'sourceDataUpdateDate', 'buildDate', 'gjDownloadURL')},
            'caveat': a['caveat']}


def join_points_to_admin(lon, lat, admin):
    """covers(point) against every polygon (STRtree candidates). 1 hit = mapped; 0 = unmapped; >1 = ambiguous_border. Never a nearest centroid."""
    from shapely import STRtree
    from shapely.geometry import Point
    polys, T = admin['polys'], admin['table']
    tree = STRtree(polys)
    codes, status, nhit = [], [], []
    for x, y in zip(lon, lat):
        pt = Point(float(x), float(y))
        hits = [int(i) for i in tree.query(pt) if polys[int(i)].covers(pt)]
        nhit.append(len(hits))
        if len(hits) == 1:
            codes.append(float(T['province_code'].iloc[hits[0]]))
            status.append('mapped')
        else:
            codes.append(np.nan)
            status.append('unmapped' if not hits else 'ambiguous_border')
    return pd.DataFrame({'province_code': codes, 'province_join_status': status, 'n_covering_polygons': nhit})


In [4]:
# FUNCTIONS (3/6): SELECTED+LINKED event cohort with reconciled exclusion categories, sparse positive daily/6h counts and the virtual complete conditional population (pure iterators; no dense panel).
EVENT_REQUIRED = ['event_uid', 'ce_year', 'event_date', 'time_block', 'province_key', 'split', 'scope', 'source_id', 'status_25m', 'status_50m', 'status_100m',
                  'segment_status', 'segment_id_2000m', 'chosen_source_oid', 'chosen_part', 'chosen_piece']


def load_events(path, cfg, linkage_manifest):
    E = pd.read_csv(path, dtype=str, keep_default_na=False, compression='gzip', usecols=lambda c: c in set(EVENT_REQUIRED))
    miss = [c for c in EVENT_REQUIRED if c not in E.columns]
    if miss:
        raise PreparationError('event linkage audit missing columns ' + ', '.join(miss))
    uid = cfg['labels']['uid']
    if (E[uid].str.strip() == '').any():
        raise PreparationError('empty event_uid')
    dup = E[uid].duplicated(keep=False)
    if dup.any():
        raise PreparationError('duplicate event_uid: ' + str(int(dup.sum())) + ' rows (e.g. ' + ', '.join(sorted(E.loc[dup, uid].unique())[:3]) + '); nothing produced')
    rec = linkage_manifest['reconciliation']
    if len(E) != int(rec['retained']):
        raise PreparationError('event rows ' + str(len(E)) + ' != accepted 18 retained ' + str(rec['retained']))
    m50 = int((E['status_50m'] == 'matched').sum())
    ma = int(((E['status_50m'] == 'matched') & (E['segment_status'] == 'assigned')).sum())
    if m50 != int(rec['status_by_tolerance']['50m']['matched']) or ma != int(rec['matched_with_segment_50m']):
        raise PreparationError('primary matched/assigned counts differ from the accepted 18 manifest')
    years = pd.to_numeric(E['ce_year'], errors='coerce')
    absent = [y for y in cfg['labels']['years'] if int((years == y).sum()) == 0]
    if absent:
        raise PreparationError('no retained records for year(s) ' + str(absent) + ': a missing input year is an error, never a zero-filled year')
    return E


def select_events(E, inv, uni_ids, uns, cfg):
    """One category per retained record. 'selected' = primary 50 m matched AND assigned AND segment in the universe AND date in the axis.
    Every other record keeps an explicit unknown/outside category; none is allocated as a zero event."""
    lb, t = cfg['labels'], cfg['time']
    seg = E[lb['segment_column']]
    cat = np.full(len(E), '', dtype=object)

    def put(mask, lab):
        nonlocal cat
        mask = np.asarray(mask, bool) & (cat == '')
        cat = np.where(mask, lab, cat)
    put(E['scope'] != 'pilot', 'outside_pilot_envelope')
    put(E[lb['primary_status_column']] != lb['primary_status_value'], 'unknown_status_50m_' + E[lb['primary_status_column']].astype(str))
    put(E[lb['segment_status_column']] != lb['segment_status_value'], 'matched_50m_no_segment_assignment')
    open_ = cat == ''
    if (seg[open_].str.strip() == '').any():
        raise PreparationError('assigned primary matches without a 2000 m segment id')
    inv2000 = inv[inv['resolution_m'] == float(cfg['universe']['resolution_m'])].set_index('segment_id')
    missing = open_ & ~seg.isin(inv2000.index).to_numpy()
    if missing.any():
        raise PreparationError(str(int(missing.sum())) + ' assigned events point at segments absent from the accepted inventory')
    unr = dict(zip(uns['segment_id'], uns['unsupported_reason']))
    put(open_ & seg.isin(unr).to_numpy(), 'unsupported_segment_' + seg.map(unr).fillna('').astype(str))
    d = pd.to_datetime(E['event_date'], format='%Y-%m-%d', errors='coerce')
    put(d.isna() | (d < pd.Timestamp(t['population_start'])) | (d > pd.Timestamp(t['population_end'])), 'date_outside_population_axis')
    sel = cat == ''
    if (sel & ~seg.isin(set(uni_ids)).to_numpy()).any():
        raise PreparationError('internal: selected segment outside the universe')
    bad_block = sel & ~E[lb['block_column']].isin(t['time_blocks']).to_numpy()
    if bad_block.any():
        raise PreparationError(str(int(bad_block.sum())) + ' selected records have no valid 6 h block; accepted repaired times are required (no guessing)')
    s = E[sel]
    ids = inv2000.loc[s[lb['segment_column']]]
    for ec, ic in (('chosen_source_oid', 'oid'), ('chosen_part', 'part'), ('chosen_piece', 'piece')):
        if not (integral_ids(s[ec], ec) == ids[ic].to_numpy()).all():
            raise PreparationError('selected event ' + ec + ' differs from the segment geometry identity')
    if not (s['source_id'].to_numpy() == ids['source_id'].to_numpy()).all():
        raise PreparationError('selected event source differs from the segment source')
    cat = np.where(sel, 'selected', cat)
    out = E.assign(selection=cat)
    if len(out) != len(E) or (out['selection'] == '').any():
        raise PreparationError('selection reconciliation failed')
    return out


def sparse_counts(selected, cfg):
    """Positive cells only, unique keys, integer >= 1. Daily == sum of its 6 h blocks; totals == selected records."""
    lb = cfg['labels']
    k = selected.rename(columns={lb['segment_column']: 'segment_id', lb['date_column']: 'target_date', lb['block_column']: 'time_block'})
    daily = k.groupby(['segment_id', 'target_date']).size().rename('y_count').reset_index()
    blk = k.groupby(['segment_id', 'target_date', 'time_block']).size().rename('y_count').reset_index()
    daily = daily.sort_values(['target_date', 'segment_id']).reset_index(drop=True)
    blk = blk.sort_values(['target_date', 'segment_id', 'time_block']).reset_index(drop=True)
    s = blk.groupby(['segment_id', 'target_date'])['y_count'].sum()
    checks = {'daily_keys_unique': not daily.duplicated(['segment_id', 'target_date']).any(),
              'block_keys_unique': not blk.duplicated(['segment_id', 'target_date', 'time_block']).any(),
              'counts_integer_ge_1': bool((daily['y_count'] >= 1).all() and (blk['y_count'] >= 1).all()),
              'daily_equals_sum_of_blocks': bool(s.reindex(pd.MultiIndex.from_frame(daily[['segment_id', 'target_date']])).to_numpy().tolist() == daily['y_count'].tolist()),
              'daily_total_equals_selected': int(daily['y_count'].sum()) == len(selected),
              'block_total_equals_selected': int(blk['y_count'].sum()) == len(selected)}
    bad = [c for c, v in checks.items() if not v]
    if bad:
        raise PreparationError('sparse count reconciliation failed: ' + str(bad))
    return daily, blk, checks


# ---------- virtual complete conditional population (date-major, segment, block) ----------
POPULATION_UNITS = ('daily', '6h')
SPARSE_MAX_COUNT = 2 ** 53  # positive counts above this are refused as overflow (never truncated)


def _unit_blocks(ax, unit):
    if not isinstance(unit, str) or unit not in POPULATION_UNITS:
        raise PreparationError('population unit must be one of ' + str(list(POPULATION_UNITS)) + ', got ' + repr(unit))
    return len(ax['blocks']) if unit == '6h' else 1


def _axis_evidence(ax):
    return {'segments': len(ax['segments']), 'first_segment': ax['segments'][0], 'last_segment': ax['segments'][-1],
            'dates': len(ax['dates']), 'first_date': ax['dates'][0], 'last_date': ax['dates'][-1], 'blocks': list(ax['blocks'])}


def _checked_positions(pos, n, unit):
    """Validate a position vector BEFORE any cast or index: 1-D, numeric (no bool/str/object), finite, whole, 0 <= pos < n. Empty -> empty."""
    if isinstance(pos, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in pos):
        raise PreparationError('population positions must not be booleans')
    a = np.asarray(pos)
    if a.ndim != 1:
        raise PreparationError('population positions must be a 1-D vector, got ndim ' + str(a.ndim))
    if a.size == 0:
        return np.zeros(0, np.int64)
    if a.dtype.kind not in 'iuf':
        raise PreparationError('population positions must be integer numbers (bool/str/object/overflowing values refused), got dtype ' + str(a.dtype))
    if a.dtype.kind == 'f':
        if not np.isfinite(a).all():
            raise PreparationError('population positions contain NaN or infinity')
        if (a != np.floor(a)).any():
            raise PreparationError('population positions must be whole numbers (no truncation)')
    out = (a < 0) | (a >= n)
    if out.any():
        raise PreparationError(str(int(out.sum())) + ' ' + str(unit) + ' position(s) outside the defined population [0, ' + str(n) + '): an unknown cell is never a conditional zero')
    return a.astype(np.int64)


def population_axes(segment_ids, start, end, blocks):
    """Fixed axes. Refuses empty/duplicate segment ids or blocks and empty, reversed or invalid (non YYYY-MM-DD) date intervals."""
    seg = list(segment_ids)
    if not seg or any(not isinstance(s, str) or s.strip() == '' for s in seg) or len(set(seg)) != len(seg):
        raise PreparationError('population segments must be unique, non-empty string ids (at least one)')
    seg = sorted(seg)
    blk = list(blocks)
    if not blk or any(not isinstance(b, str) or b.strip() == '' for b in blk) or len(set(blk)) != len(blk):
        raise PreparationError('population time blocks must be unique, non-empty strings (at least one)')
    bounds = []
    for v in (start, end):
        t = pd.to_datetime(v, format='%Y-%m-%d', errors='coerce') if isinstance(v, str) else pd.NaT
        if pd.isna(t) or t.strftime('%Y-%m-%d') != v:
            raise PreparationError('population date bound must be a valid YYYY-MM-DD string, got ' + repr(v))
        bounds.append(t)
    if bounds[1] < bounds[0]:
        raise PreparationError('population date interval is reversed: ' + start + ' > ' + end)
    dates = pd.date_range(bounds[0], bounds[1], freq='D').strftime('%Y-%m-%d').tolist()
    return {'segments': seg, 'dates': dates, 'blocks': blk, 'seg_pos': {s: i for i, s in enumerate(seg)}, 'date_pos': {d: i for i, d in enumerate(dates)}}


def population_size(ax, unit):
    return len(ax['dates']) * len(ax['segments']) * _unit_blocks(ax, unit)


def keys_from_positions(ax, unit, pos):
    """Linear position -> keys. Order: date, then segment (sorted), then block (6 h). Positions are bound-checked first (no negative wrap, no truncation)."""
    nb = _unit_blocks(ax, unit)
    pos = _checked_positions(pos, population_size(ax, unit), unit)
    S = len(ax['segments'])
    di, rem = pos // (S * nb), pos % (S * nb)
    si, bi = rem // nb, rem % nb
    out = pd.DataFrame({'segment_id': np.asarray(ax['segments'], dtype=object)[si], 'target_date': np.asarray(ax['dates'], dtype=object)[di]})
    if unit == '6h':
        out['time_block'] = np.asarray(ax['blocks'], dtype=object)[bi]
    return out


def positions_from_keys(ax, unit, keys):
    nb = _unit_blocks(ax, unit)
    si = keys['segment_id'].map(ax['seg_pos'])
    di = keys['target_date'].map(ax['date_pos'])
    if si.isna().any() or di.isna().any():
        raise PreparationError('keys outside the population axes')
    bi = keys['time_block'].map({b: i for i, b in enumerate(ax['blocks'])}) if unit == '6h' else pd.Series(0, index=keys.index)
    if bi.isna().any():
        raise PreparationError('time_block outside the population axis')
    return (di.astype(np.int64) * len(ax['segments']) + si.astype(np.int64)) * nb + bi.astype(np.int64)


def sparse_index(ax, unit, counts):
    """Sorted positions + counts of the positive cells (for O(log n) lookups) plus the population bound/unit/axis evidence lookup_counts requires.
    Counts are validated before any integer cast (finite, whole, 1..SPARSE_MAX_COUNT); keys must be unique and inside the axes. Empty = all-zero within the bound."""
    n = population_size(ax, unit)
    raw = np.asarray(counts['y_count'])
    if len(raw):
        if raw.ndim != 1 or raw.dtype.kind not in 'iuf':
            raise PreparationError('positive counts must be integer numbers (bool/str/object/overflowing values refused), got dtype ' + str(raw.dtype))
        if raw.dtype.kind == 'f' and (~np.isfinite(raw) | (raw != np.floor(raw))).any():
            raise PreparationError('positive counts must be finite whole numbers (no truncation)')
        if ((raw < 1) | (raw > SPARSE_MAX_COUNT)).any():
            raise PreparationError('positive counts must be in 1..' + str(SPARSE_MAX_COUNT) + ' (zero/negative/overflow refused)')
    p = positions_from_keys(ax, unit, counts).to_numpy(np.int64)
    if len(p) and ((p < 0) | (p >= n)).any():
        raise PreparationError('internal: positive cell outside the population bound')
    o = np.argsort(p, kind='stable')
    p, v = p[o], raw.astype(np.int64)[o]
    if len(p) and (np.diff(p) == 0).any():
        raise PreparationError('duplicate positive cells')
    return {'pos': p, 'y': v, 'unit': unit, 'size': n, 'axis': _axis_evidence(ax)}


def lookup_counts(sp, pos):
    """y for population positions inside sp's recorded bound: the stored positive count, else 0 (0 = no SELECTED+LINKED record in that defined cell only).
    Positions outside the bound (negative, >= size, non-integral, NaN/inf, bool) raise: an unknown cell is never returned as 0."""
    miss = [k for k in ('pos', 'y', 'unit', 'size', 'axis') if k not in sp]
    if miss or sp['unit'] not in POPULATION_UNITS:
        raise PreparationError('sparse index lacks population bound evidence ' + str(miss) + ': build it with sparse_index')
    pos = _checked_positions(pos, int(sp['size']), sp['unit'])
    i = np.searchsorted(sp['pos'], pos)
    hit = (i < len(sp['pos'])) & (sp['pos'][np.minimum(i, max(len(sp['pos']) - 1, 0))] == pos) if len(sp['pos']) else np.zeros(len(pos), bool)
    y = np.zeros(len(pos), np.int64)
    y[hit] = sp['y'][i[hit]]
    return y


def iter_population_batches(ax, unit, batch_rows, date_start=None, date_end=None):
    """Validated eagerly, then a pure generator over the virtual population (optionally an inclusive date sub-range inside the axis), in fixed
    deterministic order and <= batch_rows rows. Yields (first_position, keys DataFrame). Outside or reversed sub-ranges raise (never an empty success)."""
    if isinstance(batch_rows, (bool, np.bool_)) or not isinstance(batch_rows, (int, np.integer)) or not 0 < int(batch_rows) <= 100000:
        raise PreparationError('batch_rows must be an integer in 1..100000')
    batch_rows = int(batch_rows)
    nb = _unit_blocks(ax, unit)
    per_day = len(ax['segments']) * nb
    d0 = 0 if date_start is None else ax['date_pos'].get(date_start)
    d1 = len(ax['dates']) - 1 if date_end is None else ax['date_pos'].get(date_end)
    if d0 is None or d1 is None:
        raise PreparationError('population date sub-range ' + repr((date_start, date_end)) + ' is outside the axis ' + ax['dates'][0] + '..' + ax['dates'][-1])
    if d1 < d0:
        raise PreparationError('population date sub-range is reversed: ' + str(date_start) + ' > ' + str(date_end))
    return _population_batches(ax, unit, batch_rows, d0 * per_day, (d1 + 1) * per_day)


def _population_batches(ax, unit, batch_rows, lo, hi):
    for a in range(lo, hi, batch_rows):
        b = min(a + batch_rows, hi)
        yield a, keys_from_positions(ax, unit, np.arange(a, b, dtype=np.int64))


def population_audit(ax, unit, sp, batch_rows, n_selected, splits):
    """Stream the whole virtual population once: batch bounds, first/last keys, row count and count sum per split == selected records."""
    if sp.get('unit') != unit or sp.get('size') != population_size(ax, unit) or sp.get('axis') != _axis_evidence(ax):
        raise PreparationError(str(unit) + ' sparse index was built for a different population (unit/bound/axis evidence mismatch)')
    rows, total, ysum, first, last, maxb, per = 0, 0, 0, None, None, 0, {}
    sp_names = [(s['name'], s['start'], s['end']) for s in splits]
    for a, k in iter_population_batches(ax, unit, batch_rows):
        y = lookup_counts(sp, np.arange(a, a + len(k), dtype=np.int64))
        if first is None:
            first = k.iloc[0].to_dict()
        last = k.iloc[-1].to_dict()
        maxb = max(maxb, len(k))
        rows += len(k)
        ysum += int(y.sum())
        for name, s0, s1 in sp_names:
            m = (k['target_date'] >= s0).to_numpy() & (k['target_date'] <= s1).to_numpy()
            d = per.setdefault(name, {'cells': 0, 'events': 0, 'positive_cells': 0})
            d['cells'] += int(m.sum())
            d['events'] += int(y[m].sum())
            d['positive_cells'] += int((y[m] > 0).sum())
        total += 1
    ok = rows == population_size(ax, unit) and ysum == int(n_selected) and maxb <= int(batch_rows) and int(sp['y'].sum()) == int(n_selected)
    if not ok:
        raise PreparationError(unit + ' population audit failed: rows ' + str(rows) + ', count sum ' + str(ysum) + ' vs selected ' + str(n_selected))
    return {'unit': unit, 'rows': rows, 'batches': total, 'max_batch_rows': maxb, 'count_sum': ysum, 'positive_cells': int(len(sp['pos'])),
            'first_key': first, 'last_key': last, 'by_split': per}


In [5]:
# FUNCTIONS (4/6): prospective context tables. Calendar/festival/season/bank-as-of and weather values come from the hash-verified notebook-10 helpers (h10);
# this cell adds independent guards, reasons and the DOH route/control-section AADT proxy. Factorised tables only (no date x segment duplication).
WX_REQUIRED = ['province_code', 'date', 'rain_sum', 'temp_mean', 'rain_missing', 'temp_missing', 'n_valid_rain', 'n_valid_temp', 'n_expected',
               'latest_issue_bound_utc', 'issue_bound_ok']
TEMP_PLAUSIBLE_C = (-20.0, 55.0)


def issue_times(dates, cfg):
    t = cfg['time']
    d = pd.to_datetime(pd.Series(list(dates)), format='%Y-%m-%d')
    loc = d - pd.Timedelta(days=int(t['issue_days_before_target'])) + pd.Timedelta(hours=int(t['issue_hour_local']))
    off = int(t['utc_offset_seconds'])
    sign = '+' if off >= 0 else '-'
    suffix = sign + '%02d:%02d' % (abs(off) // 3600, abs(off) % 3600 // 60)
    return loc.dt.strftime('%Y-%m-%dT%H:%M:%S') + suffix, (loc - pd.Timedelta(seconds=off)).dt.strftime('%Y-%m-%dT%H:%M:%S') + '+00:00'


def split_name(dates, cfg):
    out = np.full(len(dates), 'beyond_evaluated_period', dtype=object)
    d = np.asarray(list(dates), dtype=object)
    for s in cfg['time']['splits']:
        out = np.where((d >= s['start']) & (d <= s['end']), s['name'], out)
    return out


def date_table(h10, cfg, festivals, cal_records, cal_sources):
    """Every research date 2021-01-01..2027-12-31: target-date calendar features (notebook-10 helpers, unchanged as-of rules) + metadata."""
    t, f = cfg['time'], cfg['features']
    dates = pd.date_range(t['research_start'], t['research_end'], freq='D').strftime('%Y-%m-%d').tolist()
    core = h10['core_date_features'](dates)
    fest = h10['festival_features'](dates, festivals, int(f['festival_clip_days']), {k: tuple(v) for k, v in f['festival_windows'].items()})
    season = h10['season_features'](dates, f['season_months'])
    bank = h10['bank_calendar_features'](dates, cal_records, cal_sources, int(t['issue_hour_local']), int(t['issue_days_before_target']))
    D = core
    for part in (fest, season, bank):
        if part['target_date'].tolist() != dates:
            raise PreparationError('notebook-10 helper changed the date order')
        D = pd.concat([D, part.drop(columns=['target_date'])], axis=1)
    for c in D.columns:
        if c not in ('target_date', 'bank_calendar_coverage'):
            D[c] = D[c].astype(float)
    D['issue_time_local'], D['issue_time_utc'] = issue_times(dates, cfg)
    D['split'] = split_name(dates, cfg)
    D['in_population_axis'] = (D['target_date'] >= t['population_start']) & (D['target_date'] <= t['population_end'])
    D['beyond_evaluated_period'] = D['target_date'] > t['evaluated_end']
    return D


def date_feature_names(cfg, festivals):
    names = ['day_of_week', 'month', 'day_of_month', 'is_weekend', 'doy_sin', 'doy_cos']
    for fe in festivals:
        names += ['fest_' + fe['id'] + '_dist', 'fest_' + fe['id'] + '_in'] + ['fest_' + fe['id'] + '_' + w for w in cfg['features']['festival_windows']]
    names += ['season_' + k for k in cfg['features']['season_months']] + ['bank_holiday_asof']
    return names


def guard_weather_frame(w, valid_codes, label, blocks):
    """Independent checks of the cached notebook-09 quality flags BEFORE the notebook-10 helper: a variable flagged complete must have
    n_valid == n_expected and a finite value; codes must be registered provinces; keys unique. Inconsistency fails closed."""
    miss = [c for c in WX_REQUIRED if c not in w.columns]
    if miss:
        raise PreparationError(label + ': missing columns ' + ', '.join(miss))
    w = w.copy()
    if 'time_block' not in w.columns:
        raise PreparationError(label + ': missing time_block')
    if not w['time_block'].isin(blocks).all():
        raise PreparationError(label + ': time_block outside ' + str(blocks))
    code = integral_ids(w['province_code'], label + ' province_code')
    if not set(code.tolist()) <= set(valid_codes):
        raise PreparationError(label + ': province codes outside the registered 77')
    for v, n in (('rain', 'n_valid_rain'), ('temp', 'n_valid_temp')):
        complete = ~as_flag(w[v + '_missing'])
        nv, ne = pd.to_numeric(w[n], errors='coerce').to_numpy(float), pd.to_numeric(w['n_expected'], errors='coerce').to_numpy(float)
        val = pd.to_numeric(w[v + ('_sum' if v == 'rain' else '_mean')], errors='coerce').to_numpy(float)
        bad = complete & ((nv != ne) | ~np.isfinite(val))
        if bad.any():
            raise PreparationError(label + ': ' + str(int(bad.sum())) + ' rows flag ' + v + ' complete but have missing hours/values (cached flag inconsistent)')
    if w.duplicated(['province_code', 'date', 'time_block']).any():
        raise PreparationError(label + ': duplicate province/date/block rows')
    return w


def weather_table(w, h10, cfg, valid_codes, kind):
    """kind 'daily' -> wx_*; '6h' -> wx_block_*. Values from h10.weather_features (complete hours + both issue guards); NaN otherwise, with reasons."""
    t, wc = cfg['time'], cfg['weather']
    blocks = [wc['daily_block_label']] if kind == 'daily' else t['time_blocks']
    w = guard_weather_frame(w, valid_codes, 'weather_' + kind, blocks)
    prefix = 'wx_' if kind == 'daily' else 'wx_block_'
    out, audit = h10['weather_features'](w, {c: c for c in valid_codes}, int(wc['buffer_hours']), int(t['issue_hour_local']),
                                        int(t['issue_days_before_target']), int(t['utc_offset_seconds']), prefix)
    out = out.rename(columns={'province_key': 'province_code'}).reset_index(drop=True)
    w = w.reset_index(drop=True)
    tgt = pd.to_datetime(out['target_date'])
    issue_utc = (tgt - pd.Timedelta(days=int(t['issue_days_before_target'])) + pd.Timedelta(hours=int(t['issue_hour_local']))
                 - pd.Timedelta(seconds=int(t['utc_offset_seconds']))).dt.tz_localize('UTC')
    bound = pd.to_datetime(w['latest_issue_bound_utc'], utc=True, errors='coerce')
    guard = (as_flag(w['issue_bound_ok']) & ((bound + pd.Timedelta(hours=int(wc['buffer_hours']))) <= issue_utc).fillna(False).to_numpy())
    rc, tc = prefix + 'rain_sum_mm_d2', prefix + 'temp_mean_c_d2'
    temp = out[tc].to_numpy(float)
    implaus = np.isfinite(temp) & ((temp < TEMP_PLAUSIBLE_C[0]) | (temp > TEMP_PLAUSIBLE_C[1]))
    out.loc[implaus, tc] = np.nan
    for v, col, extra in (('rain', rc, None), ('temp', tc, implaus)):
        r = np.where(~guard, 'issue_bound_after_issue_time', np.where(as_flag(w[v + '_missing']), v + '_incomplete_hours', 'ok'))
        if extra is not None:
            r = np.where((r == 'ok') & extra, 'temp_out_of_plausible_range', r)
        r = np.where((r == 'ok') & out[col].isna().to_numpy(), v + '_value_missing', r)
        out[prefix + v + '_reason'] = r
    if ((out[prefix + 'rain_reason'] == 'ok') != out[rc].notna()).any() or ((out[prefix + 'temp_reason'] == 'ok') != out[tc].notna()).any():
        raise PreparationError('weather reasons disagree with the notebook-10 helper values')
    out['province_code'] = out['province_code'].astype(np.int64)
    if kind == 'daily':
        out = out.drop(columns=['time_block'])
    audit.update({'kind': kind, 'temp_out_of_plausible_range': int(implaus.sum())})
    return out, audit


# ---------- DOH AADT survey-point proxy ----------
def aadt_tables(st, cfg, h10):
    """Station rows (row_class station) grouped by strictly normalised DOH route + control section and survey year. Returns (groups, availability, audit)."""
    t = cfg['traffic']
    need = ['survey_year_ce', 'route', 'control_section', 'survey_point', 'province_text', 'row_class', 'total_aadt', 'total_valid', 'heavy_share_pct',
            'heavy_share_valid', 'available_from_proxy_utc']
    miss = [c for c in need if c not in st.columns]
    if miss:
        raise PreparationError('traffic stations missing columns ' + ', '.join(miss))
    a = {'rows': len(st), 'by_row_class': st['row_class'].value_counts().to_dict()}
    s = st[st['row_class'] == 'station'].copy()
    s['survey_year_ce'] = integral_ids(s['survey_year_ce'], 'survey_year_ce')
    s['aadt_route'] = [norm_numeric_code(v, t['route_regex'], t['pad']) for v in s['route']]
    s['aadt_section'] = [norm_numeric_code(v, t['route_regex'], t['pad']) for v in s['control_section']]
    bad_id = s['aadt_route'].isna() | s['aadt_section'].isna()
    a['station_rows_identity_nonstandard'] = int(bad_id.sum())
    s = s[~bad_id]
    if s.duplicated(['survey_year_ce', 'aadt_route', 'aadt_section', 'survey_point', 'province_text']).any():
        raise PreparationError('traffic: duplicate station rows for the same year/route/section/survey point/province')
    tot, hv = pd.to_numeric(s['total_aadt'], errors='coerce'), pd.to_numeric(s['heavy_share_pct'], errors='coerce')
    tv = as_flag(s['total_valid'])
    hvv = as_flag(s['heavy_share_valid'])
    if (tv & ~(np.isfinite(tot) & (tot >= 0)).to_numpy()).any() or (hvv & ~(np.isfinite(hv) & (hv >= 0) & (hv <= 100)).to_numpy()).any():
        raise PreparationError('traffic: a value flagged valid is non-finite / negative / heavy share outside 0..100 (cached flag inconsistent)')
    s['tot'], s['hv'] = tot.where(tv), hv.where(hvv)
    av = pd.to_datetime(s['available_from_proxy_utc'], utc=True, errors='coerce')
    if av.isna().any():
        raise PreparationError('traffic: station rows without a parseable available_from_proxy_utc')
    s['avail'] = av
    A = s.groupby('survey_year_ce')['avail'].agg(['min', 'max', 'nunique']).reset_index()
    A['available_from_proxy_utc'] = A['max'].map(lambda x: x.isoformat())  # conservative: the latest proxy of the vintage
    g = s.groupby(['aadt_route', 'aadt_section', 'survey_year_ce'])
    G = pd.DataFrame({'aadt_mean_total': g['tot'].mean(), 'aadt_median_total': g['tot'].median(), 'aadt_mean_heavy_share_pct': g['hv'].mean(),
                      'aadt_n_station_rows': g.size(), 'aadt_n_valid_total': g['tot'].count()}).reset_index()
    G['survey_year_ce'] = G['survey_year_ce'].astype(np.int64)
    a.update({'station_rows_used': len(s), 'groups': len(G), 'vintages': A[['survey_year_ce', 'available_from_proxy_utc', 'nunique']].to_dict('records')})
    return G, A[['survey_year_ce', 'available_from_proxy_utc']], a


def aadt_vintage_by_date(avail, dates, cfg, h10):
    """notebook-10 rule: newest survey year < target year whose proxy available_from <= issue (UTC); never the target year or a future vintage."""
    t = cfg['time']
    meta = avail.assign(normalize_status='normalized')
    v = h10['traffic_vintage_by_date'](meta, list(dates), int(t['issue_hour_local']), int(t['issue_days_before_target']), int(t['utc_offset_seconds']))
    if v['target_date'].tolist() != list(dates):
        raise PreparationError('vintage helper changed the date order')
    y = pd.to_numeric(v['trf_vintage_survey_year_ce'], errors='coerce')
    tyear = pd.to_numeric(v['target_date'].str[:4])
    if (y >= tyear).any():
        raise PreparationError('AADT vintage not strictly prior to the target year')
    return pd.DataFrame({'target_date': v['target_date'], 'aadt_vintage_survey_year_ce': y.astype(float)})


In [6]:
# FUNCTIONS (5/6): THE shared feature path for notebook 20 (training/evaluation) and 22 (serving): prepared-context tables, load_prepared_context,
# build_feature_rows, column allowlist, deterministic reference requests and leakage negative controls. No labels, no network, no fitting here.
CONTEXT_FILES = {'settings': 'context_settings.json', 'segments': 'universe_segments.csv', 'unsupported': 'unsupported_segments.csv', 'dates': 'date_features.csv',
                 'wx_daily': 'weather_context_daily.csv.gz', 'wx_6h': 'weather_context_6h.csv.gz', 'aadt': 'aadt_groups.csv'}
SEG_STR = ['segment_id', 'route_code', 'province_join_status', 'aadt_route', 'aadt_section', 'aadt_identity_status']
DATE_STR = ['target_date', 'bank_calendar_coverage', 'issue_time_local', 'issue_time_utc', 'split']
WX_DAILY = ['wx_rain_sum_mm_d2', 'wx_temp_mean_c_d2']
WX_BLOCK = ['wx_block_rain_sum_mm_d2', 'wx_block_temp_mean_c_d2']
AADT = ['aadt_mean_total', 'aadt_median_total', 'aadt_mean_heavy_share_pct']


def feature_columns(settings, unit, variant):
    """Approved model-input columns in their fixed order. route_code / time_block are categorical text; every other column is float64."""
    if unit not in UNITS or variant not in settings['variants']:
        raise RequestError('invalid_unit', 'unit must be daily|6h and variant one of ' + str(settings['variants']))
    cols = list(STATIC_FEATURES) + list(settings['date_feature_columns']) + (['time_block'] if unit == '6h' else [])
    if variant == 'calendar_geometry_context':
        cols += WX_DAILY + (WX_BLOCK if unit == '6h' else []) + AADT
    return cols


def validate_columns(cols, settings):
    """Every model input must be allowlisted and never a forbidden key/label/metadata name or substring."""
    allow = set(feature_columns(settings, '6h', 'calendar_geometry_context'))
    forb = [c for c in cols if c in set(settings['forbidden_names']) or any(s in c for s in settings['forbidden_substrings'])]
    extra = [c for c in cols if c not in allow]
    if forb or extra or len(set(cols)) != len(cols):
        raise ValueError('feature guard: forbidden ' + str(forb) + ', not allowlisted ' + str(extra))
    return True


def context_settings(cfg, festivals):
    t, f = cfg['time'], cfg['features']
    s = {'version': cfg['version'], 'snapshot_tag': cfg['snapshot_tag'], 'time_blocks': t['time_blocks'], 'research_start': t['research_start'],
         'research_end': t['research_end'], 'evaluated_end': t['evaluated_end'], 'population_start': t['population_start'], 'population_end': t['population_end'],
         'max_rows': int(cfg['batch']['max_rows']), 'variants': f['variants'], 'date_feature_columns': date_feature_names(cfg, festivals),
         'forbidden_names': f['forbidden_names'], 'forbidden_substrings': f['forbidden_substrings'], 'timezone': t['timezone']}
    validate_columns(feature_columns(s, '6h', 'calendar_geometry_context'), s)
    return s


def context_from_tables(settings, segments, unsupported, dates, wx_daily, wx_6h, aadt):
    """In-memory lookups; used identically by the notebook (fresh tables) and by load_prepared_context (files)."""
    seg = segments.set_index('segment_id', verify_integrity=True)
    dts = dates.set_index('target_date', verify_integrity=True)
    wd = wx_daily.set_index(['province_code', 'target_date'], verify_integrity=True)
    wb = wx_6h.set_index(['province_code', 'target_date', 'time_block'], verify_integrity=True)
    ag = aadt.set_index(['aadt_route', 'aadt_section', 'survey_year_ce'], verify_integrity=True)
    return {'settings': settings, 'seg': seg, 'unsupported': set(unsupported['segment_id']), 'dates': dts, 'wx_daily': wd, 'wx_6h': wb, 'aadt': ag}


def load_prepared_context(data_dir, expected_hashes):
    """Read the notebook-19 prepared tables from data_dir after each file's sha256 equals expected_hashes (the accepted manifest's
    data_files_gitignored). Numeric columns are parsed with float_precision='round_trip' (read_numeric_csv). Fails closed on any gap."""
    d = Path(data_dir)
    for key, name in CONTEXT_FILES.items():
        if name not in (expected_hashes or {}):
            raise PreparationError('no expected hash for ' + name + ' (fail closed)')
        verify_pinned(d / name, expected_hashes[name], 'prepared ' + name)
    settings = json.loads((d / CONTEXT_FILES['settings']).read_text('utf-8'))
    seg = read_numeric_csv(d / CONTEXT_FILES['segments'], SEG_STR)
    uns = read_numeric_csv(d / CONTEXT_FILES['unsupported'], ['segment_id', 'source_id', 'route_key', 'route_status', 'unsupported_reason'])
    dts = read_numeric_csv(d / CONTEXT_FILES['dates'], DATE_STR)
    for c in ('in_population_axis', 'beyond_evaluated_period'):
        dts[c] = as_flag(dts[c])
    wd = read_numeric_csv(d / CONTEXT_FILES['wx_daily'], ['target_date', 'wx_rain_reason', 'wx_temp_reason'])
    wb = read_numeric_csv(d / CONTEXT_FILES['wx_6h'], ['target_date', 'time_block', 'wx_block_rain_reason', 'wx_block_temp_reason'])
    ag = read_numeric_csv(d / CONTEXT_FILES['aadt'], ['aadt_route', 'aadt_section'])
    for df in (wd, wb):
        df['province_code'] = integral_ids(df['province_code'], 'province_code')
    ag['survey_year_ce'] = integral_ids(ag['survey_year_ce'], 'survey_year_ce')
    return context_from_tables(settings, seg, uns, dts, wd, wb, ag)


def validate_requests(ctx, requests, unit):
    """Strict request contract: exact fields, string values, ISO dates within the research range, protocol blocks, unique keys, bounded size,
    known universe segments. Raises RequestError (never returns partial or fake-zero rows). No paths are accepted from requests."""
    s = ctx['settings']
    if unit not in UNITS:
        raise RequestError('invalid_unit', 'unit must be one of ' + str(UNITS))
    R = pd.DataFrame(requests) if not isinstance(requests, pd.DataFrame) else requests.copy()
    need = ['segment_id', 'target_date'] + (['time_block'] if unit == '6h' else [])
    if sorted(R.columns) != sorted(need):
        raise RequestError('invalid_fields', 'fields must be exactly ' + str(need) + ', got ' + str(list(R.columns)))
    if len(R) == 0 or len(R) > int(s['max_rows']):
        raise RequestError('batch_size', 'rows must be 1..' + str(s['max_rows']))
    R = R[need].reset_index(drop=True)
    for c in need:
        bad = ~R[c].map(lambda v: isinstance(v, str) and v.strip() == v and v != '')
        if bad.any():
            raise RequestError('invalid_value', c + ' must be non-empty trimmed text', np.flatnonzero(bad.to_numpy())[:10])
    ok = R['target_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}') & pd.to_datetime(R['target_date'], format='%Y-%m-%d', errors='coerce').notna()
    if (~ok).any():
        raise RequestError('invalid_date', 'target_date must be an ISO local date', np.flatnonzero(~ok.to_numpy())[:10])
    out = (R['target_date'] < s['research_start']) | (R['target_date'] > s['research_end'])
    if out.any():
        raise RequestError('date_out_of_range', 'supported research dates ' + s['research_start'] + '..' + s['research_end'], np.flatnonzero(out.to_numpy())[:10])
    if unit == '6h' and (~R['time_block'].isin(s['time_blocks'])).any():
        raise RequestError('invalid_time_block', 'time_block must be one of ' + str(s['time_blocks']), np.flatnonzero(~R['time_block'].isin(s['time_blocks']).to_numpy())[:10])
    if R.duplicated(need).any():
        raise RequestError('duplicate_keys', 'request keys must be unique', np.flatnonzero(R.duplicated(need).to_numpy())[:10])
    known = R['segment_id'].isin(ctx['seg'].index)
    if (~known).any():
        uns = R['segment_id'].isin(ctx['unsupported'])
        code = 'unsupported_segment' if (uns & ~known).any() else 'unknown_segment'
        raise RequestError(code, 'segments outside the qualified DOH 2000 m universe have no prediction (grey/unsupported, never zero)',
                           np.flatnonzero((~known).to_numpy())[:10])
    return R


def build_feature_rows(ctx, requests, unit, variant='calendar_geometry_context'):
    """(features, metadata) for validated requests, row i <-> request i (positional lookups: no merge, no row multiplication).
    features: approved columns only, declared order, float64 except categorical text route_code/time_block; source gaps stay NaN.
    metadata: keys, issue time (target day - 1 at 18:00 Asia/Bangkok), period flag and per-source reasons (never model inputs)."""
    s = ctx['settings']
    R = validate_requests(ctx, requests, unit)
    cols = feature_columns(s, unit, variant)
    validate_columns(cols, s)
    si = ctx['seg'].index.get_indexer(R['segment_id'])
    di = ctx['dates'].index.get_indexer(R['target_date'])
    if (si < 0).any() or (di < 0).any():
        raise RequestError('unknown_key', 'segment/date missing from the prepared context')
    S, D = ctx['seg'].iloc[si].reset_index(), ctx['dates'].iloc[di].reset_index()
    X = pd.DataFrame(index=R.index)
    for c in STATIC_FEATURES:
        X[c] = S[c].astype(str).to_numpy() if c == 'route_code' else S[c].to_numpy(float)
    for c in s['date_feature_columns']:
        X[c] = D[c].to_numpy(float)
    if unit == '6h':
        X['time_block'] = R['time_block'].to_numpy(object)
    pc = S['province_code'].to_numpy(float)
    has_p = np.isfinite(pc)
    pci = np.where(has_p, pc, -1).astype(np.int64)
    M = pd.DataFrame({'segment_id': R['segment_id'], 'target_date': R['target_date']})
    if unit == '6h':
        M['time_block'] = R['time_block']
    M['issue_time_local'], M['issue_time_utc'] = D['issue_time_local'].to_numpy(), D['issue_time_utc'].to_numpy()
    M['split'], M['beyond_evaluated_period'] = D['split'].to_numpy(), D['beyond_evaluated_period'].to_numpy(bool)
    M['bank_calendar_known_asof'], M['bank_calendar_coverage'] = D['bank_calendar_known_asof'].to_numpy(float), D['bank_calendar_coverage'].to_numpy()
    M['province_code'], M['province_join_status'] = pc, S['province_join_status'].to_numpy()
    if variant == 'calendar_geometry_context':
        prov_reason = np.where(has_p, '', 'segment_province_' + S['province_join_status'].astype(str).to_numpy())
        groups = [(ctx['wx_daily'], pd.MultiIndex.from_arrays([pci, R['target_date']]), 'wx_')]
        if unit == '6h':
            groups.append((ctx['wx_6h'], pd.MultiIndex.from_arrays([pci, R['target_date'], R['time_block']]), 'wx_block_'))
        for tab, key, pre in groups:
            ix = tab.index.get_indexer(key)
            for v, col in (('rain', pre + 'rain_sum_mm_d2'), ('temp', pre + 'temp_mean_c_d2')):
                val = np.where(ix >= 0, tab[col].to_numpy(float)[np.maximum(ix, 0)], np.nan) if len(tab) else np.full(len(R), np.nan)
                rs = np.where(ix >= 0, tab[pre + v + '_reason'].to_numpy(object)[np.maximum(ix, 0)], 'not_in_weather_archive') if len(tab) else \
                    np.full(len(R), 'not_in_weather_archive', dtype=object)
                rs = np.where(has_p, rs, prov_reason)
                X[col] = np.where(has_p, val, np.nan)
                M[pre + v + '_reason'] = rs
        vint = D['aadt_vintage_survey_year_ce'].to_numpy(float)
        ident = S['aadt_identity_status'].astype(str).to_numpy()
        vi = np.where(np.isfinite(vint), vint, -1).astype(np.int64)
        ix = ctx['aadt'].index.get_indexer(pd.MultiIndex.from_arrays([S['aadt_route'].astype(str).to_numpy(), S['aadt_section'].astype(str).to_numpy(), vi]))
        usable = (ix >= 0) & (ident == 'ok') & np.isfinite(vint)
        for c in AADT + ['aadt_n_station_rows']:
            arr = np.where(usable, ctx['aadt'][c].to_numpy(float)[np.maximum(ix, 0)], np.nan) if len(ctx['aadt']) else np.full(len(R), np.nan)
            (M if c == 'aadt_n_station_rows' else X)[c] = arr
        M['aadt_vintage_survey_year_ce'] = vint
        M['aadt_reason'] = np.where(ident != 'ok', 'segment_identity_' + ident, np.where(~np.isfinite(vint), 'no_prior_available_vintage',
                                    np.where(ix < 0, 'no_station_for_route_section_in_vintage',
                                             np.where(np.isnan(X['aadt_mean_total'].to_numpy(float)), 'no_valid_total', 'ok'))))
    X = X[cols]
    fin = [c for c in STATIC_FEATURES if c != 'route_code'] + [c for c in s['date_feature_columns'] if c != 'bank_holiday_asof']
    if not np.isfinite(X[fin].to_numpy(float)).all():
        raise PreparationError('non-finite static/calendar feature')
    num = X.drop(columns=[c for c in ('route_code', 'time_block') if c in X.columns])
    if np.isinf(num.to_numpy(float)).any() or len(X) != len(R):
        raise PreparationError('infinite feature value or row count changed')
    return X.reset_index(drop=True), M.reset_index(drop=True)


def reference_requests(ctx, unit, cfg):
    """Deterministic request sample (seeded, unique keys): segments x population dates plus a share of beyond-evaluated dates (2026-27)."""
    r, s = cfg['reference'], ctx['settings']
    rng = np.random.default_rng(int(r['seed']) + (0 if unit == 'daily' else 1))
    n = int(r['rows_per_unit'])
    nb = int(round(n * float(r['beyond_period_share'])))
    seg = np.asarray(sorted(ctx['seg'].index), dtype=object)
    dts = ctx['dates']
    pop = np.asarray(dts.index[dts['in_population_axis'].to_numpy(bool)], dtype=object)
    beyond = np.asarray(dts.index[dts['beyond_evaluated_period'].to_numpy(bool)], dtype=object)
    out = []
    for k, pool in ((n - nb, pop), (nb, beyond)):
        m = k * 3
        q = pd.DataFrame({'segment_id': seg[rng.integers(0, len(seg), m)], 'target_date': pool[rng.integers(0, len(pool), m)]})
        if unit == '6h':
            q['time_block'] = np.asarray(s['time_blocks'], dtype=object)[rng.integers(0, len(s['time_blocks']), m)]
        out.append(q.drop_duplicates().head(k))
    R = pd.concat(out, ignore_index=True).drop_duplicates().reset_index(drop=True)
    if len(R) != n:
        raise PreparationError('reference sample could not reach ' + str(n) + ' unique keys')
    return R


def frames_identical(a, b):
    """Same columns/order/dtypes kind and identical values (NaN == NaN; floats bitwise-equal by value)."""
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return False
    for c in a.columns:
        x, y = a[c].to_numpy(), b[c].to_numpy()
        if a[c].dtype.kind == 'f' or b[c].dtype.kind == 'f':
            if not np.array_equal(x.astype(float), y.astype(float), equal_nan=True):
                return False
        elif not (pd.Series(x).astype(str).to_numpy() == pd.Series(y).astype(str).to_numpy()).all():
            return False
    return True


def leakage_negative_controls(ctx, h10, cfg):
    """Deliberately leaky or invalid inputs; every control must be REJECTED by the production functions."""
    t, s, res = cfg['time'], ctx['settings'], []
    # 1. notebook-10's own controls (future traffic vintage, late/removed bank announcements, unknown calendar, late weather, tampered guard, label columns)
    allow = feature_columns(s, '6h', 'calendar_geometry_context')
    for r in h10['leakage_negative_controls']([], allow, s['forbidden_names'], s['forbidden_substrings'], int(cfg['weather']['buffer_hours']),
                                              int(t['issue_hour_local']), int(t['issue_days_before_target']), int(t['utc_offset_seconds'])):
        res.append({'control': 'nb10_' + r['control'], 'rejected': bool(r['rejected'])})
    # 2. keys, labels, metadata and event-derived columns offered as model inputs
    for bad in ('segment_id', 'oid', 'piece', 'province_code', 'aadt_n_station_rows', 'aadt_vintage_survey_year_ce', 'bank_calendar_known_asof',
                'y_count', 'event_lat', 'wx_rain_reason', 'beyond_evaluated_period', 'split'):
        try:
            validate_columns(allow + [bad], s)
            ok = False
        except ValueError:
            ok = True
        res.append({'control': 'allowlist_rejects_' + bad, 'rejected': ok})
    # 3. AADT: same-year and future-published vintages
    av = pd.DataFrame({'survey_year_ce': [2023, 2024], 'available_from_proxy_utc': ['2024-06-01T00:00:00+00:00', '2024-01-02T00:00:00+00:00']})
    v = aadt_vintage_by_date(av, ['2024-03-01', '2024-06-03'], cfg, h10)
    res.append({'control': 'aadt_future_publication_or_same_year', 'rejected': bool(np.isnan(v['aadt_vintage_survey_year_ce'].iloc[0]))})
    res.append({'control': 'aadt_prior_year_after_publication_used', 'rejected': bool(v['aadt_vintage_survey_year_ce'].iloc[1] == 2023.0)})
    # 4. request contract: label/extra field, unknown segment, out-of-range date, bad block
    seg0 = ctx['seg'].index[0]
    bad_reqs = [('request_label_field', [{'segment_id': seg0, 'target_date': '2023-01-01', 'y_count': '0'}], 'daily'),
                ('request_unknown_segment', [{'segment_id': 'not-a-segment', 'target_date': '2023-01-01'}], 'daily'),
                ('request_date_out_of_range', [{'segment_id': seg0, 'target_date': '2028-01-01'}], 'daily'),
                ('request_invalid_block', [{'segment_id': seg0, 'target_date': '2023-01-01', 'time_block': '06-18'}], '6h'),
                ('request_invalid_unit', [{'segment_id': seg0, 'target_date': '2023-01-01'}], 'weekly')]
    for name, req, unit in bad_reqs:
        try:
            build_feature_rows(ctx, req, unit)
            ok = False
        except RequestError:
            ok = True
        res.append({'control': name, 'rejected': ok})
    return res


In [7]:
# FUNCTIONS (6/6): aggregate audits (support, coverage, as-of, province agreement), data dictionary, figures, Thai report and the fail-closed manifest.
CAVEATS = [
    'label = SELECTED+LINKED retained MOT records (status_50m matched AND segment assigned) per segment x local day / 6 h block; a zero is exact only inside this conditional target, never zero real accidents or complete reporting',
    'unknown / unmatched / unassigned / outside-envelope / unsupported-segment records are audited separately and never allocated as zero events',
    'current road snapshot applied retrospectively; historical topology, control sections and event datum are unverified',
    'geoBoundaries ADM1 (OSM-derived, year represented 2017, ODbL) is not a current official boundary; ambiguous/unmapped midpoints have no province context',
    'weather = coarse province representative-point JMA GSM previous_day2 archive (nominal issue bound + 6 h buffer); not road weather; historical delivery unverified',
    'AADT = DOH annual survey-point proxy by current route/control section from the newest prior vintage with catalog proxy availability <= issue; not live traffic or segment exposure',
    'bank calendar = reviewed BOT records as of issue; unknown coverage is NaN, never an ordinary day; bank holiday != public holiday',
    'dates 2024-2025 are previously seen diagnostics; 2026-2027 are beyond the evaluated period (features only, no labels)',
    'preparation only: no model was fitted, no score, threshold or risk colour exists']


def count_support(daily, blk, ax, cfg):
    rows = []
    for unit, C in (('daily', daily), ('6h', blk)):
        nb = len(ax['blocks']) if unit == '6h' else 1
        for s in cfg['time']['splits']:
            c = C[(C['target_date'] >= s['start']) & (C['target_date'] <= s['end'])]
            nd = sum(1 for d in ax['dates'] if s['start'] <= d <= s['end'])
            cells = nd * len(ax['segments']) * nb
            rows.append({'unit': unit, 'split': s['name'], 'date_start': s['start'], 'date_end': s['end'], 'segments': len(ax['segments']), 'days': nd,
                         'population_cells': cells, 'positive_cells': len(c), 'selected_records': int(c['y_count'].sum()),
                         'zero_cells_conditional': cells - len(c), 'max_cell_count': int(c['y_count'].max()) if len(c) else 0,
                         'segments_with_any_record': int(c['segment_id'].nunique()), 'mean_count_per_cell': float(c['y_count'].sum()) / cells if cells else np.nan})
    return pd.DataFrame(rows)


def coverage_audit(ctx, ax, unit, batch_rows, label):
    """Stream the virtual population through build_feature_rows: per-year non-missing counts of context/bank features and reason counts."""
    acc, reasons = {}, {}
    for a, k in iter_population_batches(ax, unit, batch_rows):
        X, M = build_feature_rows(ctx, k, unit)
        yr = k['target_date'].str[:4].to_numpy()
        ctxcols = ['bank_holiday_asof'] + WX_DAILY + (WX_BLOCK if unit == '6h' else []) + AADT
        for y in np.unique(yr):
            m = yr == y
            d = acc.setdefault(y, {'rows': 0, **{c: 0 for c in ctxcols}})
            d['rows'] += int(m.sum())
            for c in ctxcols:
                d[c] += int(X.loc[m, c].notna().sum())
            for rc in [c for c in M.columns if c.endswith('_reason')]:
                for rv, n in M.loc[m, rc].value_counts().items():
                    key = (y, rc, rv)
                    reasons[key] = reasons.get(key, 0) + int(n)
    cov = pd.DataFrame([{'scope': label, 'unit': unit, 'year': y, 'rows': v['rows'], 'feature': c, 'non_missing': n, 'share': n / v['rows']}
                        for y, v in sorted(acc.items()) for c, n in v.items() if c != 'rows'])
    rs = pd.DataFrame([{'scope': label, 'unit': unit, 'year': y, 'reason_column': rc, 'reason': rv, 'rows': n} for (y, rc, rv), n in sorted(reasons.items())])
    return cov, rs


def province_agreement(E_sel, seg_tab, pts):
    """Descriptive only: event province_key vs the province of its segment midpoint (never used to drop or select anything)."""
    code_to_key = dict(zip(integral_ids(pts['province_code'], 'province_code').tolist(), pts['province_key']))
    sp = seg_tab.set_index('segment_id')
    pc = sp.loc[E_sel['segment_id_2000m'], 'province_code'].to_numpy(float)
    seg_key = np.array([code_to_key.get(int(c)) if np.isfinite(c) else None for c in pc], dtype=object)
    st = np.where(seg_key == None, 'segment_midpoint_' + sp.loc[E_sel['segment_id_2000m'], 'province_join_status'].astype(str).to_numpy(),  # noqa: E711
                  np.where(seg_key == E_sel['province_key'].to_numpy(object), 'same_province', 'different_province'))
    return pd.DataFrame({'split': E_sel['split'].to_numpy(), 'agreement': st}).value_counts().rename('selected_records').reset_index()


def data_dictionary(settings):
    g = {c: ('static_geometry', 'pinned 18 2000 m GeoJSON, EPSG:32647 always_xy') for c in STATIC_FEATURES}
    g['route_code'] = ('static_route_category', 'inventory route_key (DOH zfill4); categorical, not ordinal')
    for c in settings['date_feature_columns']:
        g[c] = ('target_calendar', 'notebook-10 helpers on the target local date')
    g['bank_holiday_asof'] = ('bank_calendar_asof', 'reviewed BOT records known by target day - 1 18:00; NaN when coverage unknown')
    g['time_block'] = ('target_block', 'target 6 h block (Asia/Bangkok); categorical')
    for c in WX_DAILY + WX_BLOCK:
        g[c] = ('weather_context', 'province representative-point JMA GSM previous_day2; NaN when incomplete/late/unmapped')
    for c in AADT:
        g[c] = ('aadt_context', 'DOH survey-point AADT by route+control section, newest prior available vintage; NaN with reason')
    units = {'geom_mid_x_m': 'm (EPSG:32647)', 'geom_mid_y_m': 'm (EPSG:32647)', 'geom_mid_lon': 'deg', 'geom_mid_lat': 'deg', 'geom_length_m': 'm (projected)',
             'geom_log_length_m': 'ln(m)', 'geom_bearing_sin': 'sin(grid azimuth start->end); 0 when chord <= eps', 'geom_bearing_cos': 'cos(...); 0 when chord <= eps',
             'geom_straightness': 'chord / length, 0..1', 'geom_turning_rad_per_km': 'sum |turn| rad per km (zero-length edges skipped)',
             'wx_rain_sum_mm_d2': 'mm/day', 'wx_temp_mean_c_d2': 'deg C', 'wx_block_rain_sum_mm_d2': 'mm/6 h', 'wx_block_temp_mean_c_d2': 'deg C',
             'aadt_mean_total': 'vehicles/day (mean of survey points)', 'aadt_median_total': 'vehicles/day (median)', 'aadt_mean_heavy_share_pct': '%'}
    rows = []
    for c in feature_columns(settings, '6h', 'calendar_geometry_context'):
        rows.append({'column': c, 'role': 'model_input', 'group': g[c][0], 'dtype': 'str (categorical)' if c in ('route_code', 'time_block') else 'float64',
                     'unit': units.get(c, ''), 'source': g[c][1], 'units': 'daily+6h' if c not in WX_BLOCK + ['time_block'] else '6h',
                     'variants': 'calendar_geometry_context' if c in WX_DAILY + WX_BLOCK + AADT else 'both'})
    for c, desc in (('segment_id', 'key only (never a predictor)'), ('target_date', 'key, Asia/Bangkok local date'), ('time_block', 'key for 6 h'),
                    ('issue_time_local', 'target day - 1 at 18:00 +07:00'), ('beyond_evaluated_period', 'True after 2025-12-31'),
                    ('province_code / province_join_status', 'midpoint polygon join metadata'), ('*_reason', 'source gap reason'),
                    ('aadt_n_station_rows / aadt_vintage_survey_year_ce', 'AADT metadata'), ('bank_calendar_known_asof / bank_calendar_coverage', 'source availability metadata'),
                    ('y_count', 'label from sparse counts (0 only inside the defined conditional population)')):
        rows.append({'column': c, 'role': 'metadata_or_label_not_feature', 'group': '', 'dtype': '', 'unit': '', 'source': desc, 'units': '', 'variants': ''})
    return pd.DataFrame(rows)


def set_fonts(cfg):
    import matplotlib
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    fam = [f for f in cfg['figure_fonts'] if f in have]
    matplotlib.rcParams['font.family'] = fam + ['DejaVu Sans'] if fam else ['DejaVu Sans']


def save_fig(fig, path):
    fig.savefig(path, dpi=110, bbox_inches='tight', metadata={'Software': None})
    import matplotlib.pyplot as plt
    plt.close(fig)


def make_figures(fig_dir, daily_counts, cov, gaps, cfg):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    set_fonts(cfg)
    fig_dir.mkdir(parents=True, exist_ok=True)
    m = daily_counts.assign(month=daily_counts['target_date'].str[:7]).groupby('month')['y_count'].sum()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(range(len(m)), m.to_numpy(), color='#4C72B0')
    ax.set_xticks(range(0, len(m), 6))
    ax.set_xticklabels(m.index[::6], rotation=45)
    ax.set_ylabel('selected+linked records')
    ax.set_title('Actual SELECTED+LINKED record counts per month (DOH 2000 m universe) - preparation, NOT a forecast')
    save_fig(fig, fig_dir / 'selected_counts_by_month.png')
    fig, ax = plt.subplots(figsize=(10, 4.5))
    c = cov[cov['unit'] == 'daily'].pivot_table(index='year', columns='feature', values='share')
    for col in c.columns:
        ax.plot(c.index.astype(str), c[col].to_numpy(), marker='o', label=col)
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel('share of population cells with a value')
    ax.set_title('Context feature coverage 2021-2025 (daily population) - preparation, NOT a forecast')
    ax.legend(fontsize=7, ncol=2)
    save_fig(fig, fig_dir / 'feature_coverage_by_year.png')
    fig, ax = plt.subplots(figsize=(10, 4.5))
    g = gaps.pivot_table(index='year', columns='feature', values='share')
    g.plot(kind='bar', ax=ax)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('share with a value')
    ax.set_title('Prospective source gaps 2026-2027 (beyond evaluated period, daily) - preparation, NOT a forecast')
    ax.legend(fontsize=7, ncol=2)
    save_fig(fig, fig_dir / 'prospective_source_gaps.png')
    return ['figures/selected_counts_by_month.png', 'figures/feature_coverage_by_year.png', 'figures/prospective_source_gaps.png']


def md_table(df, max_rows=40):
    if df is None or not len(df):
        return '(ว่าง)'
    d = df.head(max_rows)
    h = '| ' + ' | '.join(str(c) for c in d.columns) + ' |'
    s = '| ' + ' | '.join('---' for _ in d.columns) + ' |'
    body = [('| ' + ' | '.join('' if (isinstance(v, float) and not math.isfinite(v)) else str(v) for v in r) + ' |') for r in d.itertuples(index=False)]
    return NL.join([h, s] + body)


def report_md(T, status, checks, cfg, out_root, preflight):
    L = ['# Notebook 19 — เตรียมข้อมูล spatial CountRegression (ไม่ใช่โมเดล ไม่ใช่การพยากรณ์)', '',
         'สถานะ: **' + status + '** · เวอร์ชัน ' + cfg['version'] + ' · snapshot ' + cfg['snapshot_tag'], '',
         '## 1. ขอบเขตเป้าหมาย (target)', cfg['labels']['meaning'], '',
         '## 2. จักรวาล segment (กำหนดจาก geometry เท่านั้น ก่อนมีคะแนนโมเดลใด)', cfg['universe']['rule'], '',
         'ค่าที่วัดจริงในรันนี้ (preflight ของผู้จัดการเป็นบริบทเท่านั้น: ' + str(preflight) + '):', md_table(T['universe_summary']), '',
         'segment ที่ไม่รองรับ (ไม่มีการทำนาย ไม่ใช่ศูนย์):', md_table(T['unsupported_summary']), '',
         '## 3. การคัดเลือกบันทึก (reconciled ทุกแถว)', md_table(T['selection_reconciliation'], 80), '',
         '## 4. จำนวนนับ sparse และประชากรเสมือน', md_table(T['count_support']), '', md_table(T['population_audit']), '',
         '## 5. จังหวัดจากจุดกึ่งกลาง segment (geoBoundaries 2017/OSM ไม่ใช่ขอบเขตทางการปัจจุบัน)', md_table(T['province_join_summary']), '',
         'ความสอดคล้องจังหวัดของเหตุกับจังหวัดของ segment (เชิงพรรณนาเท่านั้น):', md_table(T['province_agreement']), '',
         '## 6. ความครอบคลุมของ feature บริบท และ as-of', md_table(T['feature_coverage'], 60), '', md_table(T['asof_audit'], 40), '',
         '## 7. Negative controls การรั่วไหล', md_table(T['leakage_negative_controls'], 60), '',
         '## 8. Parity ของ feature builder (in-memory = reload context = CSV round-trip)', md_table(T['reference_parity']), '',
         '## 9. คำสั่งรัน', '- `.venv/bin/python backend/scripts/run_spatial_preparation.py --output-dir backend/rerun-spatial-prep-<id>`',
         '- output root ของรันนี้: `' + str(out_root) + '`', '',
         '## 10. Checks', md_table(pd.DataFrame([{'check': c['check'], 'status': c['status']} for c in checks]), 100), '',
         '## 11. ข้อจำกัด', NL.join('- ' + c for c in CAVEATS), '',
         'ยังไม่มีการฝึกโมเดล ไม่มี negative sampling ไม่มีคะแนน ไม่มีเกณฑ์สี; การผ่าน check ไม่ใช่การยอมรับผลิตภัณฑ์']
    return NL.join(L) + NL


def build_manifest(status, started, cfg, cfg_sha, nb_path, input_rows, checks, report_files, data_files, protected_before, protected_after, extra):
    changed = sorted(k for k in set(protected_before) | set(protected_after) if protected_before.get(k) != protected_after.get(k))
    return {'status': status, 'version': cfg['version'], 'purpose': 'spatial count preparation: universe, sparse SELECTED+LINKED targets, prospective features, leakage evidence; NOT a model or forecast',
            'started_utc': started, 'finished_utc': utcnow(), 'config_sha256': cfg_sha, 'notebook': cfg['notebook'],
            'notebook_sha256': sha256_file(nb_path) if Path(nb_path).is_file() else None,
            'functions_cells_sha256': extra.get('functions_cells_sha256'), 'code_cells_sha256': extra.get('code_cells_sha256'),
            'notebook10_functions_sha256': extra.get('nb10_sha'), 'inputs': input_rows, 'checks': checks,
            'environment': {'python': platform.python_version(), 'platform': platform.platform(),
                            'dependencies': {m: module_version(m) for m in ('pandas', 'numpy', 'shapely', 'pyproj', 'matplotlib')}},
            'report_files': report_files, 'data_files_gitignored': data_files, 'protected_files_hashed': len(protected_before),
            'protected_unchanged': not changed, 'protected_changed': changed, 'protected_sha256_before': protected_before,
            'readiness': {'production_ready': False, 'model_trained': False, 'next': 'notebook 20 protocol lock + training per protocol (not started)', 'caveats': CAVEATS},
            **{k: v for k, v in extra.items() if k not in ('functions_cells_sha256', 'code_cells_sha256', 'nb10_sha')}}


def notebook_cell_hashes(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    out, funcs = {}, []
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] == 'code':
            s = ''.join(c['source']) if isinstance(c['source'], list) else c['source']
            out[str(i) + ':' + str(c.get('id', ''))] = sha256_bytes(s.encode('utf-8'))
            if s.lstrip().startswith('# FUNCTIONS'):
                funcs.append(s)
    return sha256_bytes('\n\n'.join(funcs).encode('utf-8')), out


In [8]:
# SETUP: config validation, output guard BEFORE any write, protected hashes before, check registry and the fail-closed (readable failed status) stage guard
import contextlib

STARTED = utcnow()
cfg = json.loads(CONFIG_PATH.read_text('utf-8'))
CFG_SHA = sha256_file(CONFIG_PATH)
validate_config(cfg)
require_modules(['pandas', 'numpy', 'shapely', 'pyproj', 'matplotlib'], cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
check_outputs(PATHS, BACKEND, cfg)  # raises before anything is created
NB_PATH = BACKEND / cfg['notebook']
I = cfg['inputs']
PROTECTED_BEFORE = hash_paths(BACKEND, protected_rels(cfg))
CHECKS, T, EXTRA, DATA_FILES = [], {}, {}, {}
MANIFEST_PATH = PATHS['report_dir'] / 'spatial_preparation_manifest.json'


def check(name, ok, evidence=None):
    CHECKS.append({'check': name, 'status': 'pass' if bool(ok) else 'fail', 'evidence': evidence})
    if not bool(ok):
        raise PreparationError('check failed: ' + name + ' | ' + str(evidence)[:800])


@contextlib.contextmanager
def stage(name):
    try:
        yield
    except Exception as e:
        CHECKS.append({'check': 'stage ' + name + ' completed', 'status': 'fail', 'evidence': type(e).__name__ + ': ' + str(e)[:3000]})
        PATHS['report_dir'].mkdir(parents=True, exist_ok=True)
        atomic_write_json(MANIFEST_PATH, {'status': 'failed', 'failed_stage': name, 'error': type(e).__name__ + ': ' + str(e)[:3000], 'version': cfg['version'],
                                          'started_utc': STARTED, 'finished_utc': utcnow(), 'config_sha256': CFG_SHA, 'checks': CHECKS,
                                          'note': 'fail closed: no success manifest; outputs in this root are incomplete evidence only'})
        raise


print('outputs:', PATHS['report_dir'], '|', PATHS['data_dir'], '| protected files hashed:', len(PROTECTED_BEFORE))


outputs: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/rerun-spatial-prep-015-002/reports/v3/spatial_preparation | /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/rerun-spatial-prep-015-002/data/v3/spatial_preparation | protected files hashed: 634


In [9]:
# 1. Inputs: manager pins == producer manifests, accepted 18 status, verified notebook-10 FUNCTIONS helpers, calendar policy parity
with stage('inputs'):
    INPUT_DF, DOCS = verify_inputs(BACKEND, cfg)
    h10, NB10_SHA = load_nb10_helpers(BACKEND / I['notebook10']['path'], I['notebook10']['functions_sha256'], I['notebook10']['functions_cells'],
                                      DOCS['preparation_v2_manifest'])
    INPUT_DF = pd.concat([INPUT_DF, pd.DataFrame([{'input': 'notebook10_functions', 'path': I['notebook10']['path'], 'sha256': NB10_SHA,
                                                   'rule': 'FUNCTIONS sha == manager pin == V2 preparation manifest', 'status': 'pass'}])], ignore_index=True)
    check('every pinned input equals its manager pin and producer-manifest hash', (INPUT_DF['status'] == 'pass').all(), {'inputs': len(INPUT_DF)})
    FCFG, SRC_CFG = DOCS['feature_config_v2'], DOCS['sources_config_v2']
    FESTIVALS = SRC_CFG['calendar']['recurring_fixed_festivals']
    CAL_SOURCES = SRC_CFG['calendar']['sources']
    t, fz = cfg['time'], cfg['features']
    parity = {'clip': FCFG['festivals']['clip_days'] == fz['festival_clip_days'],
              'windows': {k: list(v) for k, v in FCFG['festivals']['windows'].items()} == {k: list(v) for k, v in fz['festival_windows'].items()},
              'season': FCFG['season']['months'] == fz['season_months'], 'issue_hour': FCFG['panel']['issue_hour_local'] == t['issue_hour_local'],
              'days_before': FCFG['panel']['issue_days_before_target'] == t['issue_days_before_target'],
              'utc_offset': FCFG['panel']['utc_offset_seconds'] == t['utc_offset_seconds'], 'blocks': FCFG['panel']['time_blocks'] == t['time_blocks'],
              'weather_buffer': FCFG['weather']['buffer_hours'] == cfg['weather']['buffer_hours']}
    check('calendar/festival/season/issue/weather-buffer policy identical to the accepted notebook-10 config (no changed as-of rules)', all(parity.values()), parity)
    FUNC_SHA, CELL_SHAS = notebook_cell_hashes(NB_PATH)
    EXTRA.update(functions_cells_sha256=FUNC_SHA, code_cells_sha256=CELL_SHAS, nb10_sha=NB10_SHA)
    TR = make_transformers(cfg)
    T['input_verification'] = INPUT_DF
INPUT_DF


,input,path,sha256,rule,status
0,protocol_v3,../workflow/spatial-model-protocol-v3-draft.md,b286fd5ed7b914bad32b88aec666388b87db2880b51ab0...,manager pin,pass
1,linkage_manifest,reports/v3/linkage_audit/linkage_audit_manifes...,92cb1377ec4d677dea9cdc56536cf8e91c0f33bf8849bb...,manager pin,pass
2,linkage_review,reports/v3/linkage_audit_independent_review.json,811b2304a8e0ce48123ef00fd5a41b982096cdaeddd45a...,manager pin,pass
3,road_manifest,reports/v3/road_collection_v3_1/road_collectio...,6783d259ae91db2e2b684d0c3cd9ae376b70dfeb1af74f...,manager pin,pass
4,sources_manifest,reports/v2/sources/sources_manifest.json,9e3b996c6d695f8795fad7f76a6e9de81a492d97242d19...,manager pin,pass
5,preparation_v2_manifest,reports/v2/preparation/preparation_manifest.json,d385a9b3e3d4ded037ca34f4f8626ea1020a738d2aeac0...,manager pin,pass
6,feature_config_v2,config/feature_pipeline_v2.json,dd217247db12c8c9c5d30d2770ce368b8f30a6984a9fb1...,manager pin,pass
7,sources_config_v2,config/external_sources_v2.json,742c25b21e7a7c36f9e80a3609a54d09142ccec8a64138...,manager pin,pass
8,event_linkage,data/v3/linkage_audit/event_linkage_audit.csv.gz,bed011650ee7682426c8925230ab1c26db3c458da09398...,manager pin == linkage_manifest,pass
9,segment_inventory,data/v3/linkage_audit/segment_inventory.csv.gz,99200d276c6990a6f4b0a445f43c058d4c5f56b2925ebf...,manager pin == linkage_manifest,pass


In [10]:
# 2. Geometry-only universe (fixed before any score), static features, ADM1 registry, then midpoint province join
with stage('universe'):
    INV = load_inventory(BACKEND / I['segment_inventory']['path'])
    GEO = load_segment_geojson(BACKEND / I['segments_geojson']['path'])
    INV2000 = INV[INV['resolution_m'] == float(cfg['universe']['resolution_m'])]
    check('2000 m GeoJSON feature ids equal the 2000 m inventory ids (stable 18 IDs, no re-segmentation)', set(GEO) == set(INV2000['segment_id']),
          {'geojson': len(GEO), 'inventory_2000m': len(INV2000)})
    UNI, UNS = classify_segments(INV, cfg)
    check('every 2000 m inventory segment is either in the universe or explicitly unsupported', len(UNI) + len(UNS) == len(INV2000) and len(UNI) > 0,
          {'universe': len(UNI), 'unsupported': len(UNS)})
    STATIC = static_geometry(UNI, GEO, cfg, TR)
    IDENT = doh_identity(BACKEND / I['doh_full']['path'], STATIC['oid'], STATIC['route_code'], cfg)
    ADMIN = load_admin(BACKEND / I['adm1_geometry']['path'], BACKEND / I['adm1_metadata']['path'], BACKEND / I['province_mapping']['path'], cfg,
                       I['adm1_geometry']['sha256'])
    check('ADM1 registered before the join: 77 valid unrepaired polygons, unique ISO/ID, ISO <-> province_points mapping consistent',
          len(ADMIN['table']) == int(cfg['admin']['expected_polygons']), ADMIN['metadata'])
    JOIN = join_points_to_admin(STATIC['geom_mid_lon'], STATIC['geom_mid_lat'], ADMIN)
    SEG = pd.concat([STATIC.drop(columns=['coords_ll']).reset_index(drop=True), IDENT, JOIN], axis=1)
    check('universe segment ids unique and equal to the geometry predicate (zero-record segments included)',
          SEG['segment_id'].is_unique and set(SEG['segment_id']) == set(UNI['segment_id']), {'segments': len(SEG)})
    PREFLIGHT = cfg['universe']['preflight_context']
    T['universe_summary'] = pd.DataFrame([{
        'segments': len(SEG), 'routes': int(SEG['route_code'].nunique()), 'projected_km': round(float(SEG['geom_length_m'].sum()) / 1000, 3),
        'geodesic_km': round(float(SEG['length_geod_m'].sum()) / 1000, 3), 'utm_over_geodesic_min': float(SEG['utm_over_geodesic'].min()),
        'utm_over_geodesic_max': float(SEG['utm_over_geodesic'].max()), 'short_final_segments': int(as_flag(SEG['short_final']).sum()),
        'piece_overlap_segments': int(as_flag(SEG['piece_overlap']).sum()), 'piece_duplicate_segments': int(as_flag(SEG['piece_duplicate']).sum()),
        'chord_zero_guard_segments': int(SEG['chord_zero_guard'].sum()), 'aadt_identity_ok': int((SEG['aadt_identity_status'] == 'ok').sum()),
        'preflight_segments_context': PREFLIGHT['segments'], 'preflight_routes_context': PREFLIGHT['routes'], 'preflight_km_context': PREFLIGHT['projected_km']}])
    T['unsupported_summary'] = UNS.groupby(['source_id', 'unsupported_reason']).agg(segments=('segment_id', 'size'), km=('length_m', lambda s: round(s.sum() / 1000, 3))).reset_index()
    T['province_join_summary'] = SEG.groupby(['province_join_status', 'n_covering_polygons']).agg(segments=('segment_id', 'size'),
                                                                                                     km=('geom_length_m', lambda s: round(s.sum() / 1000, 3))).reset_index()
    T['static_feature_quality'] = SEG[[c for c in STATIC_FEATURES if c != 'route_code']].describe().T.reset_index().rename(columns={'index': 'feature'})
T['universe_summary']


,segments,routes,projected_km,geodesic_km,utm_over_geodesic_min,utm_over_geodesic_max,short_final_segments,piece_overlap_segments,piece_duplicate_segments,chord_zero_guard_segments,aadt_identity_ok,preflight_segments_context,preflight_routes_context,preflight_km_context
0,3105,218,5900.233,5899.307,0.999806,1.000897,403,0,0,0,3105,3105,218,5900.233


In [11]:
# 3. SELECTED+LINKED cohort, reconciled exclusions, sparse daily/6h counts and the virtual population audit (streamed, <= 100000-row batches)
with stage('labels'):
    E = load_events(BACKEND / I['event_linkage']['path'], cfg, DOCS['linkage_manifest'])
    ES = select_events(E, INV, SEG['segment_id'], UNS, cfg)
    SEL = ES[ES['selection'] == 'selected'].copy()
    check('one selection category per retained record (selected + every exclusion == retained)', len(ES) == len(E) and int(ES['selection'].value_counts().sum()) == len(E),
          ES['selection'].value_counts().to_dict())
    only100 = (ES['status_100m'] == 'matched') & (ES['status_50m'] != 'matched')
    check('selected records are primary 50 m matched AND assigned; 100 m-only matches never selected',
          bool(((SEL['status_50m'] == 'matched') & (SEL['segment_status'] == 'assigned')).all()) and not (only100 & (ES['selection'] == 'selected')).any(),
          {'selected': len(SEL), 'only_100m_matched_excluded': int(only100.sum())})
    yrs = pd.to_numeric(SEL['event_date'].str[:4])
    check('every label year has selected records (a missing year fails closed, never a zero-filled year)',
          all(int((yrs == y).sum()) > 0 for y in cfg['labels']['years']), yrs.value_counts().sort_index().to_dict())
    DAILY, BLK, CC = sparse_counts(SEL, cfg)
    for k, v in CC.items():
        check('sparse counts: ' + k, v)
    AX = population_axes(SEG['segment_id'], cfg['time']['population_start'], cfg['time']['population_end'], cfg['time']['time_blocks'])
    SP = {'daily': sparse_index(AX, 'daily', DAILY), '6h': sparse_index(AX, '6h', BLK)}
    POP = {u: population_audit(AX, u, SP[u], cfg['batch']['population_batch_rows'], len(SEL), cfg['time']['splits']) for u in UNITS}
    check('virtual population: rows == segments x dates (x blocks), count sum == selected records, batches <= limit',
          all(p['count_sum'] == len(SEL) for p in POP.values()), {u: {k: p[k] for k in ('rows', 'batches', 'max_batch_rows', 'count_sum')} for u, p in POP.items()})
    T['selection_reconciliation'] = ES.groupby(['ce_year', 'split', 'selection']).size().rename('records').reset_index()
    T['count_support'] = count_support(DAILY, BLK, AX, cfg)
    T['population_audit'] = pd.DataFrame([{'unit': u, 'rows': p['rows'], 'batches': p['batches'], 'max_batch_rows': p['max_batch_rows'], 'count_sum': p['count_sum'],
                                           'positive_cells': p['positive_cells'], 'first_key': json.dumps(p['first_key']), 'last_key': json.dumps(p['last_key'])}
                                          for u, p in POP.items()])
    PTS = pd.read_csv(BACKEND / I['province_mapping']['path'], dtype=str, keep_default_na=False)
    T['province_agreement'] = province_agreement(SEL, SEG, PTS)
    UNK = ES[ES['selection'] != 'selected'][['event_uid', 'ce_year', 'split', 'event_date', 'time_block', 'province_key', 'scope', 'status_50m',
                                              'segment_status', 'segment_id_2000m', 'selection']]
T['count_support']


,unit,split,date_start,date_end,segments,days,population_cells,positive_cells,selected_records,zero_cells_conditional,max_cell_count,segments_with_any_record,mean_count_per_cell
0,daily,train,2021-01-01,2023-12-31,3105,1095,3399975,14448,15213,3385527,4,1928,0.004474
1,daily,seen_2024,2024-01-01,2024-12-31,3105,366,1136430,5699,6015,1130731,6,1320,0.005293
2,daily,seen_2025,2025-01-01,2025-12-31,3105,365,1133325,5917,6231,1127408,6,1391,0.005498
3,6h,train,2021-01-01,2023-12-31,3105,1095,13599900,14805,15213,13585095,4,1928,0.001119
4,6h,seen_2024,2024-01-01,2024-12-31,3105,366,4545720,5868,6015,4539852,6,1320,0.001323
5,6h,seen_2025,2025-01-01,2025-12-31,3105,365,4533300,6099,6231,4527201,6,1391,0.001374


In [12]:
# 4. Context tables: target-date calendar (+ bank as of issue), province-point weather (daily, 6 h), DOH AADT vintages; as-of audits and negative controls
with stage('context'):
    CAL = h10['load_calendar_records'](BACKEND / I['calendar_records']['path'])
    DATES = date_table(h10, cfg, FESTIVALS, CAL, CAL_SOURCES)
    ST = pd.read_csv(BACKEND / I['traffic_stations']['path'], dtype=str, keep_default_na=False)
    AG, AV, AADT_AUDIT = aadt_tables(ST, cfg, h10)
    VINT = aadt_vintage_by_date(AV, DATES['target_date'], cfg, h10)
    DATES = DATES.merge(VINT, on='target_date', how='left', validate='one_to_one')
    CODES = sorted(int(c) for c in ADMIN['table']['province_code'])
    WXD, WXD_AUDIT = weather_table(pd.read_csv(BACKEND / I['weather_daily']['path'], dtype=str, keep_default_na=False), h10, cfg, CODES, 'daily')
    WX6, WX6_AUDIT = weather_table(pd.read_csv(BACKEND / I['weather_6h']['path'], dtype=str, keep_default_na=False), h10, cfg, CODES, '6h')
    SETTINGS = context_settings(cfg, FESTIVALS)
    CTX_MEM = context_from_tables(SETTINGS, SEG, UNS, DATES, WXD, WX6, AG)
    # independent as-of re-checks
    av = dict(zip(AV['survey_year_ce'].astype(int), pd.to_datetime(AV['available_from_proxy_utc'], utc=True)))
    iu = pd.to_datetime(DATES['issue_time_utc'], utc=True)
    vy = DATES['aadt_vintage_survey_year_ce']
    late = [d for d, y, i in zip(DATES['target_date'], vy, iu) if np.isfinite(y) and (int(y) >= int(d[:4]) or av[int(y)] > i)]
    check('AADT vintage strictly prior to the target year and available by the issue time (no future/current-year backfill)', not late, {'violations': late[:5]})
    known = DATES['bank_calendar_known_asof'] == 1.0
    check('bank holiday is NaN exactly when reviewed coverage is unknown at issue (never an ordinary day)',
          bool((DATES['bank_holiday_asof'].isna() == ~known).all()), DATES.groupby(DATES['target_date'].str[:4])['bank_calendar_coverage'].first().to_dict())
    rows = []
    for y, g in DATES.groupby(DATES['target_date'].str[:4]):
        rows.append({'source': 'bank_calendar', 'year': y, 'detail': json.dumps(g['bank_calendar_coverage'].value_counts().to_dict(), ensure_ascii=False)})
        rows.append({'source': 'aadt_vintage', 'year': y, 'detail': json.dumps({str(k): int(v) for k, v in g['aadt_vintage_survey_year_ce'].fillna(-1).astype(int).value_counts().sort_index().items()})})
    for a in (WXD_AUDIT, WX6_AUDIT):
        rows.append({'source': 'weather_' + a['kind'], 'year': 'all', 'detail': json.dumps(jsonable(a))})
    rows.append({'source': 'aadt_stations', 'year': 'all', 'detail': json.dumps(jsonable(AADT_AUDIT), ensure_ascii=False)})
    T['asof_audit'] = pd.DataFrame(rows)
    NEG = pd.DataFrame(leakage_negative_controls(CTX_MEM, h10, cfg))
    check('every leakage / invalid-request negative control is rejected by the production functions', bool(NEG['rejected'].all()),
          NEG.loc[~NEG['rejected'], 'control'].tolist())
    T['leakage_negative_controls'] = NEG
T['asof_audit']


,source,year,detail
0,bank_calendar,2021,"{""reviewed_annual_base_known_later_amendments_..."
1,aadt_vintage,2021,"{""-1"": 365}"
2,bank_calendar,2022,"{""reviewed_annual_base_known_later_amendments_..."
3,aadt_vintage,2022,"{""-1"": 84, ""2021"": 281}"
4,bank_calendar,2023,"{""reviewed_annual_base_known_later_amendments_..."
5,aadt_vintage,2023,"{""2021"": 365}"
6,bank_calendar,2024,"{""reviewed_annual_base_known_later_amendments_..."
7,aadt_vintage,2024,"{""2021"": 153, ""2022"": 213}"
8,bank_calendar,2025,"{""reviewed_annual_base_known_later_amendments_..."
9,aadt_vintage,2025,"{""2022"": 150, ""2024"": 215}"


In [13]:
# 5. Ignored data files, reload through load_prepared_context, reference feature parity (memory == reloaded context == CSV round-trip), full-population coverage
with stage('data_and_parity'):
    DD = PATHS['data_dir']

    def put(name, data):
        atomic_write_bytes(DD / name, data)
        DATA_FILES[name] = sha256_file(DD / name)

    put('context_settings.json', (json.dumps(jsonable(SETTINGS), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8'))
    put('universe_segments.csv', csv_bytes(SEG))
    put('unsupported_segments.csv', csv_bytes(UNS))
    put('date_features.csv', csv_bytes(DATES))
    put('weather_context_daily.csv.gz', csv_bytes(WXD, gz=True))
    put('weather_context_6h.csv.gz', csv_bytes(WX6, gz=True))
    put('aadt_groups.csv', csv_bytes(AG))
    put('aadt_vintage_availability.csv', csv_bytes(AV))
    put('counts_daily.csv.gz', csv_bytes(DAILY, gz=True))
    put('counts_6h.csv.gz', csv_bytes(BLK, gz=True))
    put('population_axes.json', (json.dumps({'order': 'date, then segment_id (sorted), then time_block (6h)', 'segments': AX['segments'], 'dates': AX['dates'],
                                             'blocks': AX['blocks'], 'label': cfg['labels']['meaning']}, ensure_ascii=False) + NL).encode('utf-8'))
    put('selected_events.csv.gz', csv_bytes(SEL[['event_uid', 'ce_year', 'split', 'event_date', 'time_block', 'province_key', 'segment_id_2000m',
                                                 'chosen_source_oid', 'chosen_part', 'chosen_piece']], gz=True))
    put('unknown_scope_events.csv.gz', csv_bytes(UNK, gz=True))
    put('universe_geometry.geojson', json.dumps({'type': 'FeatureCollection', 'note': 'EPSG:4326 lon/lat reference geometry of the qualified universe; snapshot ' + cfg['snapshot_tag'],
                                                 'features': [{'type': 'Feature', 'id': sid, 'properties': {'route_code': rc},
                                                               'geometry': {'type': 'LineString', 'coordinates': [[float(x), float(y)] for x, y in ll]}}
                                                              for sid, rc, ll in zip(STATIC['segment_id'], STATIC['route_code'], STATIC['coords_ll'])]},
                                                ensure_ascii=False, allow_nan=False).encode('utf-8'))
    CTX = load_prepared_context(DD, DATA_FILES)
    par = []
    for unit in UNITS:
        REQ = reference_requests(CTX, unit, cfg)
        X1, M1 = build_feature_rows(CTX_MEM, REQ, unit)
        X2, M2 = build_feature_rows(CTX, REQ, unit)
        cols = feature_columns(SETTINGS, unit, 'calendar_geometry_context')
        put('reference_requests_' + unit + '.csv', csv_bytes(REQ))
        put('reference_features_' + unit + '.csv', csv_bytes(pd.concat([M1.drop(columns=['time_block'], errors='ignore'), X1], axis=1)))
        RL = read_numeric_csv(DD / ('reference_features_' + unit + '.csv'), ['segment_id', 'target_date', 'time_block', 'route_code'])
        X3 = RL[cols]
        perm = np.random.default_rng(7).permutation(len(REQ))
        X4, _ = build_feature_rows(CTX, REQ.iloc[perm].reset_index(drop=True), unit)
        Xb, _ = build_feature_rows(CTX, REQ, unit, 'calendar_geometry')
        res = {'unit': unit, 'rows': len(REQ), 'beyond_evaluated_rows': int(M1['beyond_evaluated_period'].sum()),
               'memory_equals_reloaded_context': frames_identical(X1, X2) and frames_identical(M1, M2),
               'memory_equals_csv_round_trip': frames_identical(X1, X3),
               'permuted_requests_permute_rows': frames_identical(X4, X1.iloc[perm].reset_index(drop=True)),
               'base_variant_is_prefix_subset': frames_identical(Xb, X1[feature_columns(SETTINGS, unit, 'calendar_geometry')]),
               'columns': len(cols), 'nan_cells': int(X1.drop(columns=[c for c in ('route_code', 'time_block') if c in X1.columns]).isna().sum().sum())}
        par.append(res)
        check('reference parity ' + unit + ': same builder, reload and CSV round-trip identical, order preserved, no multiplication',
              res['memory_equals_reloaded_context'] and res['memory_equals_csv_round_trip'] and res['permuted_requests_permute_rows'] and res['base_variant_is_prefix_subset'], res)
    T['reference_parity'] = pd.DataFrame(par)
    covs, reasons = [], []
    if cfg['batch']['full_population_feature_audit']:
        for unit in UNITS:
            c, r = coverage_audit(CTX, AX, unit, cfg['batch']['population_batch_rows'], 'population_2021_2025')
            covs.append(c)
            reasons.append(r)
    AXB = population_axes(SEG['segment_id'], '2026-01-01', cfg['time']['research_end'], cfg['time']['time_blocks'])
    GAPS, GR = coverage_audit(CTX, AXB, 'daily', cfg['batch']['population_batch_rows'], 'beyond_2026_2027')
    T['feature_coverage'] = pd.concat(covs + [GAPS], ignore_index=True)
    T['source_gap_reasons'] = pd.concat(reasons + [GR], ignore_index=True)
T['reference_parity']


,unit,rows,beyond_evaluated_rows,memory_equals_reloaded_context,memory_equals_csv_round_trip,permuted_requests_permute_rows,base_variant_is_prefix_subset,columns,nan_cells
0,daily,2000,200,True,True,True,True,38,4392
1,6h,2000,200,True,True,True,True,41,6351


In [14]:
# 6. Aggregate reports, figures, Thai report, data dictionary, readiness; protected inputs re-hashed; success manifest written LAST
with stage('reports'):
    PROTECTED_AFTER = hash_paths(BACKEND, protected_rels(cfg))
    changed = sorted(k for k in set(PROTECTED_BEFORE) | set(PROTECTED_AFTER) if PROTECTED_BEFORE.get(k) != PROTECTED_AFTER.get(k))
    check('protected inputs/outputs unchanged (sha256 before == after)', not changed, {'files_hashed': len(PROTECTED_BEFORE), 'changed': changed[:20]})
    check('source notebook and config unchanged during the run', sha256_file(CONFIG_PATH) == CFG_SHA and notebook_cell_hashes(NB_PATH)[0] == EXTRA['functions_cells_sha256'])
    RD = PATHS['report_dir']
    T['data_dictionary'] = data_dictionary(SETTINGS)
    T['checks'] = pd.DataFrame([{'check': c['check'], 'status': c['status']} for c in CHECKS])
    REPORT_FILES = {}
    for name, df in T.items():
        check_report_file(name + '.csv', cfg)
        atomic_write_bytes(RD / (name + '.csv'), csv_bytes(df))
    figs = make_figures(PATHS['figure_dir'], DAILY, T['feature_coverage'][T['feature_coverage']['scope'] == 'population_2021_2025'], GAPS, cfg)
    readiness = {'status': 'preparation_checks_passed_pending_codex_review', 'production_ready': False, 'model_trained': False,
                 'feature_variants': SETTINGS['variants'], 'units': list(UNITS), 'builder': 'build_feature_rows(load_prepared_context(data_dir, manifest.data_files_gitignored), requests, unit, variant)',
                 'caveats': CAVEATS}
    atomic_write_json(RD / 'preparation_readiness.json', readiness)
    md = report_md(T, 'complete (preparation only)', CHECKS, cfg, OUTPUT_ROOT, cfg['universe']['preflight_context'])
    atomic_write_bytes(RD / 'spatial_preparation_report_th.md', md.encode('utf-8'))
    for p in sorted(RD.rglob('*')):
        if p.is_file() and p.name != MANIFEST_PATH.name:
            check_report_file(p.name, cfg)
            REPORT_FILES[p.relative_to(RD).as_posix()] = sha256_file(p)
    for name in DATA_FILES:
        check('data file unchanged since write: ' + name, sha256_file(PATHS['data_dir'] / name) == DATA_FILES[name])
    extra = dict(EXTRA, output_root=str(OUTPUT_ROOT), snapshot_tag=cfg['snapshot_tag'],
                 summary={'universe_segments': len(SEG), 'routes': int(SEG['route_code'].nunique()), 'projected_km': round(float(SEG['geom_length_m'].sum()) / 1000, 3),
                          'unsupported_2000m_segments': len(UNS), 'retained_records': len(ES), 'selected_records': len(SEL),
                          'selection': ES['selection'].value_counts().to_dict(), 'positive_cells': {u: int(len(SP[u]['pos'])) for u in UNITS},
                          'population': {u: {k: POP[u][k] for k in ('rows', 'batches', 'count_sum', 'first_key', 'last_key', 'by_split')} for u in UNITS}},
                 feature_columns={u: {v: feature_columns(SETTINGS, u, v) for v in SETTINGS['variants']} for u in UNITS},
                 admin=ADMIN['metadata'], admin_caveat=ADMIN['caveat'])
    MANIFEST = build_manifest('complete', STARTED, cfg, CFG_SHA, NB_PATH, INPUT_DF.to_dict('records'), CHECKS, REPORT_FILES, DATA_FILES,
                              PROTECTED_BEFORE, PROTECTED_AFTER, extra)
    if MANIFEST['protected_changed'] or any(c['status'] != 'pass' for c in CHECKS):
        raise PreparationError('refusing to write a success manifest with failed checks')
    atomic_write_json(MANIFEST_PATH, MANIFEST)
print('status: complete |', len(CHECKS), 'checks passed |', MANIFEST_PATH)


status: complete | 41 checks passed | /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/rerun-spatial-prep-015-002/reports/v3/spatial_preparation/spatial_preparation_manifest.json


## Outputs

- `reports/v3/spatial_preparation/` (aggregates only): `spatial_preparation_manifest.json` (written last; `status: failed` with the failed stage otherwise), `preparation_readiness.json`, `spatial_preparation_report_th.md`, `data_dictionary.csv`, `checks.csv`, `input_verification.csv`, `universe_summary.csv`, `unsupported_summary.csv`, `province_join_summary.csv`, `province_agreement.csv`, `static_feature_quality.csv`, `selection_reconciliation.csv`, `count_support.csv`, `population_audit.csv`, `asof_audit.csv`, `feature_coverage.csv`, `source_gap_reasons.csv`, `leakage_negative_controls.csv`, `reference_parity.csv`, `figures/*.png`.
- `data/v3/spatial_preparation/` (gitignored, row level): prepared context (`context_settings.json`, `universe_segments.csv`, `unsupported_segments.csv`, `date_features.csv`, `weather_context_{daily,6h}.csv.gz`, `aadt_groups.csv`), `aadt_vintage_availability.csv`, sparse `counts_{daily,6h}.csv.gz`, `population_axes.json`, `selected_events.csv.gz`, `unknown_scope_events.csv.gz`, `universe_geometry.geojson`, `reference_{requests,features}_{daily,6h}.csv`.

Notebook 20/22 usage: verify the manifest (`status == complete`), then `ctx = load_prepared_context(data_dir, manifest['data_files_gitignored'])` and `X, meta = build_feature_rows(ctx, requests, unit, variant)`; labels come only from the sparse counts via `population_axes` / `sparse_index` / `lookup_counts` / `iter_population_batches`.
